# Part B â€” Testing on real Chicago & NYC taxi data (multi-year)

Runs the *same, unmodified* Stage 1â€“4 model code from Part A, fed real taxi data instead of the repo's 11-day Shenzhen sample:
- **Chicago: 2024â€“2025** (2 full years, official city API)
- **NYC: 2024â€“2026** (2 full years plus however much of 2026 TLC has published so far â€” checked live each run, since TLC adds a new month roughly every 2 months)

**Same spatial recipe as the paper**, so the network itself needs zero changes: a 40Ã—50 fine grid over each city, cut into 10Ã—10 windows sliding by 5 cells â†’ 63 regions, exactly matching Shenzhen's own count.

**Both cities now use the same kind of location data â€” centroids, not exact GPS.** Chicago's own fields (`pickup_centroid_latitude/longitude`) are already the **centroid of the pickup census tract**, not an exact address â€” that's Chicago's own privacy-rounding, built into the dataset before we touch it. NYC's data only gives a zone ID, so we join it to the **official TLC zone centroid**. Both cities are therefore gridded the same way: one representative point per trip, not the exact pickup spot.

**Train / test split.** The paper's own words (Appendix A.1): *"We train on 3 regions and select the remaining region for testing."* We do the same thing at our scale: of the 63 regions, **47 (75%) are used for training, 16 (25%) are held out and never seen during training** â€” used only to measure how well the pretrained model reconstructs data from regions it never trained on. That is the actual train/test split for this stage (Stage 1 is unsupervised masked reconstruction â€” there's no "label" to predict yet, so "test" here means *reconstruction accuracy on unseen regions*, the same thing the paper itself measures).

Still true from before: the paper never says which 12 hours of the day it kept, so this notebook uses all 24, clearly labeled as a deliberate difference.

In [ ]:
# --- Config for Part B ---
# (Part A -- the original UrbanMind repo's own Stage 1-4 demo -- and B.3-B.7 -- reproducing
# that same method on real data -- have both been removed. This pipeline is now fully
# self-contained: it never clones or runs any of that code. The only pieces of setup this
# section used to borrow from Part A are torch/device, restated directly below.)
import numpy as np
import pandas as pd
import requests, io, os
from datetime import datetime, timedelta
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA available:", torch.cuda.is_available(), "| device:", device)

GRID_ROWS, GRID_COLS = 40, 50   # same fine-grid resolution the paper used for Shenzhen
WINDOW, STRIDE = 10, 5          # same sliding-window recipe -> 63 regions, matches the repo's hardcoded 63
TIME_SLOTS_PER_DAY = 24         # see honesty note above (paper's "12" is unspecified which 12)

GLOBAL_START = datetime(2024, 1, 1)   # day 0 for BOTH cities, so years never collide in the array index
MAX_DAYS = 1200                       # generous upper bound (2024-2026); trimmed to actually-used days later

def day_index(ts_series):
    """Continuous day offset from GLOBAL_START -- unlike day-of-year, this doesn't reset each Jan 1,
    so 2024/2025/2026 all land in distinct, correctly-ordered slots of the same array."""
    return (ts_series.dt.normalize() - pd.Timestamp(GLOBAL_START)).dt.days.values

def sliding_regions(rows, cols, window, stride):
    coords = []
    for r in range(0, rows - window + 1, stride):
        for c in range(0, cols - window + 1, stride):
            coords.append((r, c))
    return coords

REGION_COORDS = sliding_regions(GRID_ROWS, GRID_COLS, WINDOW, STRIDE)
NUM_REGIONS = len(REGION_COORDS)
print(f"Regions produced by the paper's own 40x50 / 10x10 / stride-5 recipe: {NUM_REGIONS}")
assert NUM_REGIONS == 63, "Expected 63 to match the repo's hardcoded region count"

# Region-holdout split -- same idea the paper itself uses ("train on 3 regions, test on the rest")
TEST_REGION_FRAC = 0.25
N_TEST_REGIONS = round(NUM_REGIONS * TEST_REGION_FRAC)
TEST_REGION_IDX = list(range(NUM_REGIONS - N_TEST_REGIONS, NUM_REGIONS))
TRAIN_REGION_IDX = list(range(0, NUM_REGIONS - N_TEST_REGIONS))
print(f"Region-holdout split: {len(TRAIN_REGION_IDX)} train regions, {len(TEST_REGION_IDX)} test regions (held out, never trained on)")

In [ ]:
# --- Shared aggregation logic: bin trips into the 40x50 fine grid, per day-hour, then slice into 63 regions ---

def fine_grid_edges(lat_min, lat_max, lon_min, lon_max):
    lat_edges = np.linspace(lat_min, lat_max, GRID_ROWS + 1)
    lon_edges = np.linspace(lon_min, lon_max, GRID_COLS + 1)
    return lat_edges, lon_edges

def assign_cell(lat, lon, lat_edges, lon_edges):
    row = np.clip(np.digitize(lat, lat_edges) - 1, 0, GRID_ROWS - 1)
    col = np.clip(np.digitize(lon, lon_edges) - 1, 0, GRID_COLS - 1)
    return row, col

def new_accumulators():
    shape = (MAX_DAYS, TIME_SLOTS_PER_DAY, GRID_ROWS, GRID_COLS)
    return {
        "pickup": np.zeros(shape, dtype=np.float64),
        "dropoff": np.zeros(shape, dtype=np.float64),
        "speed_sum": np.zeros(shape, dtype=np.float64),
        "speed_n": np.zeros(shape, dtype=np.float64),
    }

def slice_into_regions(dense):
    """dense: (days, hours, GRID_ROWS, GRID_COLS) -> (days, hours, NUM_REGIONS, 10, 10)"""
    days, hours = dense.shape[0], dense.shape[1]
    out = np.zeros((days, hours, NUM_REGIONS, WINDOW, WINDOW), dtype=np.float32)
    for i, (r, c) in enumerate(REGION_COORDS):
        out[:, :, i, :, :] = dense[:, :, r:r+WINDOW, c:c+WINDOW]
    return out

def trim_to_used_days(*dense_arrays):
    ref = dense_arrays[0]
    used_days = np.where(ref.reshape(MAX_DAYS, -1).sum(axis=1) > 0)[0]
    d0, d1 = used_days.min(), used_days.max() + 1
    return tuple(a[d0:d1] for a in dense_arrays), d0, d1

def write_city_csvs(city_name, speed_dense, inflow_dense, demand_dense):
    """Each *_dense is (days, hours, GRID_ROWS, GRID_COLS). Writes speed/inflow/demand CSVs in
    the exact flat row-per-day format models/masking/util.py's loaddata() expects."""
    os.makedirs(f"city_data/{city_name}", exist_ok=True)
    for name, dense in [("speed", speed_dense), ("inflow", inflow_dense), ("demand", demand_dense)]:
        regions = slice_into_regions(dense)              # (days, hours, 63, 10, 10)
        days = regions.shape[0]
        flat = regions.reshape(days, -1)                 # C-order flatten per day, matches loaddata()'s reshape contract
        pd.DataFrame(flat).to_csv(f"city_data/{city_name}/{name}.csv", header=False, index=False)
        print(f"{city_name}/{name}.csv: shape {regions.shape} -> {flat.shape}")

# Re-downloading city data (or re-querying OSM for static features) every time this notebook
# reruns wastes real time/credit for data that hasn't changed. If a previous successful run's
# Output was attached as an input dataset (Kaggle: "Add Input" -> "Notebook Output Files" -> your
# own notebook, or a dedicated cache dataset), its cached files show up somewhere under
# /kaggle/input/ -- reuse them and skip the network calls entirely. Falls back to a real fetch if
# nothing is found (e.g. the very first run).
import glob

def zone_area_matrix(geoms, lat_edges, lon_edges, samples=40000, seed=42):
    """For each zone polygon, the fraction of its AREA falling in each grid cell.

    Chicago and NYC publish trip locations identified by zone (census tract / taxi zone) rather
    than by true coordinate, so a trip is only known to have occurred SOMEWHERE in its zone. Its
    demand is therefore distributed across the grid in proportion to how much of the zone's area
    falls in each cell. Porto needs none of this -- its feed carries real per-trip GPS.

    The distribution is uniform over area ON PURPOSE: weighting it by POI or building density
    would construct the prediction target out of the model's own input features, which is
    circular and would inflate measured accuracy.

    Known limitation: a geographically large zone whose demand is genuinely concentrated in one
    corner (Chicago's O'Hare tract, ~22% of the city's trips) is distributed over its whole
    footprint.
    """
    from shapely import contains_xy
    rng = np.random.default_rng(seed)
    n_cols = len(lon_edges) - 1
    W = np.zeros((len(geoms), (len(lat_edges) - 1) * n_cols), dtype=np.float64)
    for i, geom in enumerate(geoms):
        if geom is None or geom.is_empty:
            continue
        minx, miny, maxx, maxy = geom.bounds
        lats, lons, tries = [], [], 0
        while len(lats) < samples and tries < samples * 60:
            xs = rng.uniform(minx, maxx, 20000)
            ys = rng.uniform(miny, maxy, 20000)
            m = contains_xy(geom, xs, ys)
            lats.extend(ys[m].tolist()); lons.extend(xs[m].tolist())
            tries += 20000
        if not lats:                      # degenerate polygon -> fall back to its centroid
            c = geom.centroid
            lats, lons = [c.y], [c.x]
        r, c = assign_cell(np.array(lats[:samples]), np.array(lons[:samples]), lat_edges, lon_edges)
        counts = np.bincount(r * n_cols + c, minlength=W.shape[1]).astype(np.float64)
        W[i] = counts / counts.sum()
    return W


def spread_into(acc, day, hour, zone_idx, values, W):
    """Add `values` into acc[day, hour], spread over cells by each row's zone-area profile."""
    from scipy.sparse import coo_matrix
    if len(day) == 0:
        return
    dh = day * TIME_SLOTS_PER_DAY + hour
    uniq, inv = np.unique(dh, return_inverse=True)
    M = coo_matrix((values, (inv, zone_idx)), shape=(len(uniq), W.shape[0])).tocsr()
    spread = np.asarray(M @ W).reshape(len(uniq), acc.shape[2], acc.shape[3])
    np.add.at(acc, (uniq // TIME_SLOTS_PER_DAY, uniq % TIME_SLOTS_PER_DAY), spread)


def find_precomputed(filename):
    if os.path.exists(filename):
        return filename
    # Search any depth under /kaggle/input -- an attached dataset/notebook-output preserves
    # whatever folder structure it was saved with, so a fixed-depth glob would miss it.
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    return matches[0] if matches else None

### B.1 â€” Chicago, 2024â€“2025 (city's own centroid coordinates, official API)

Chicago publishes one dataset per year on its open data portal. 2024 and 2025 are two separate, complete datasets (each with its own ID); we pull both and stack them into one continuous timeline.

In [ ]:
# Chicago Data Portal -- one dataset ID per year (both confirmed to exist and be complete as of 2026-09).
CHI_YEARS = [2024, 2025]
CHI_DATASET_IDS = {2024: "sa9s-wkhk", 2025: "b8xg-w8bx"}
CHI_COLS = ("trip_start_timestamp,trip_seconds,trip_miles,"
            "pickup_centroid_latitude,pickup_centroid_longitude,"
            "dropoff_centroid_latitude,dropoff_centroid_longitude")

import time

def fetch_chicago_month(year, month):
    base_url = f"https://data.cityofchicago.org/resource/{CHI_DATASET_IDS[year]}.csv"
    start = datetime(year, month, 1)
    end = datetime(year + (month == 12), (month % 12) + 1, 1)
    frames = []
    cur = start
    while cur < end:
        nxt = min(cur + timedelta(days=5), end)
        where = (f"trip_start_timestamp >= '{cur.isoformat()}' AND "
                 f"trip_start_timestamp < '{nxt.isoformat()}' AND "
                 f"pickup_centroid_latitude IS NOT NULL AND dropoff_centroid_latitude IS NOT NULL")
        params = {"$select": CHI_COLS, "$where": where, "$limit": 200000}
        for attempt in range(4):
            try:
                r = requests.get(base_url, params=params, timeout=180)
                r.raise_for_status()
                break
            except (requests.exceptions.RequestException,) as e:
                if attempt == 3:
                    raise
                wait = 5 * (2 ** attempt)
                print(f"  request failed ({e}); retrying in {wait}s (attempt {attempt+1}/4)...")
                time.sleep(wait)
        df = pd.read_csv(io.StringIO(r.text))
        if len(df):
            frames.append(df)
        cur = nxt
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=CHI_COLS.split(","))

# Bounding box from real data (city limits don't change year to year, so one sample month is
# enough) -- but even that one-month sample is a real network fetch worth skipping if a cached
# chicago_dense.npz (which now also stores lat_edges/lon_edges) is already available.
CHI_DENSE_PATH = "chicago_dense.npz"
_chi_bbox_cache = find_precomputed(CHI_DENSE_PATH)
_chi_cache_has_bbox = _chi_bbox_cache and "lat_edges" in np.load(_chi_bbox_cache)
if _chi_cache_has_bbox:
    _npz_bbox = np.load(_chi_bbox_cache)
    lat_edges, lon_edges = _npz_bbox["lat_edges"], _npz_bbox["lon_edges"]
    print(f"Chicago bbox from cache ({_chi_bbox_cache}): "
          f"lat[{lat_edges[0]:.4f},{lat_edges[-1]:.4f}] lon[{lon_edges[0]:.4f},{lon_edges[-1]:.4f}]")
else:
    print("Sampling January 2024 to establish Chicago's bounding box from real data...")
    bbox_sample = fetch_chicago_month(2024, 1)
    lat_min, lat_max = bbox_sample.pickup_centroid_latitude.min(), bbox_sample.pickup_centroid_latitude.max()
    lon_min, lon_max = bbox_sample.pickup_centroid_longitude.min(), bbox_sample.pickup_centroid_longitude.max()
    lat_edges, lon_edges = fine_grid_edges(lat_min, lat_max, lon_min, lon_max)
    print(f"Chicago bbox from data: lat[{lat_min:.4f},{lat_max:.4f}] lon[{lon_min:.4f},{lon_max:.4f}]")
    del bbox_sample

In [ ]:
!pip -q install geopandas shapely pyogrio

def accumulate_chicago(df, acc):
    if df.empty:
        return
    ts = pd.to_datetime(df["trip_start_timestamp"])
    day = day_index(ts)
    hour = ts.dt.hour.values
    valid = (hour < TIME_SLOTS_PER_DAY) & (day >= 0) & (day < MAX_DAYS)
    # Chicago identifies a trip's location only by its census tract, so each trip's demand is
    # distributed across that tract's real footprint.
    pu_idx = chi_tract_index(df["pickup_centroid_latitude"].values, df["pickup_centroid_longitude"].values)
    do_idx = chi_tract_index(df["dropoff_centroid_latitude"].values, df["dropoff_centroid_longitude"].values)
    miles = df["trip_miles"].values
    secs = df["trip_seconds"].values
    speed = np.where(secs > 0, miles / (secs / 3600.0), np.nan)  # miles per hour

    valid = valid & (pu_idx >= 0) & (do_idx >= 0)
    ones = np.ones(int(valid.sum()), dtype=np.float64)
    spread_into(acc["pickup"], day[valid], hour[valid], pu_idx[valid], ones, CHI_TRACT_W)
    spread_into(acc["dropoff"], day[valid], hour[valid], do_idx[valid], ones, CHI_TRACT_W)
    ok_speed = valid & np.isfinite(speed) & (speed > 0) & (speed < 100)
    spread_into(acc["speed_sum"], day[ok_speed], hour[ok_speed], pu_idx[ok_speed],
                speed[ok_speed].astype(np.float64), CHI_TRACT_W)
    spread_into(acc["speed_n"], day[ok_speed], hour[ok_speed], pu_idx[ok_speed],
                np.ones(int(ok_speed.sum()), dtype=np.float64), CHI_TRACT_W)

# Re-downloading 24 months of Chicago data every time you re-run the notebook (e.g. after fixing
# an unrelated bug further down) wastes an hour for nothing -- the data itself hasn't changed.
# If a previous successful run's Output was attached as an input dataset (Kaggle: "Add Input" ->
# "Notebook Output Files" -> your own notebook), its cached chicago_dense.npz shows up somewhere
# under /kaggle/input/ -- reuse it and skip the download+aggregation entirely. Falls back to a
# real download if nothing is found (e.g. the very first run).
import glob

def find_precomputed(filename):
    if os.path.exists(filename):
        return filename
    # Search any depth under /kaggle/input -- an attached notebook's output preserves whatever
    # folder structure it was saved with (e.g. nested inside "UrbanMind/" here, since that's this
    # notebook's own working directory), so a fixed-depth glob would miss it.
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    return matches[0] if matches else None

CHI_DENSE_PATH = "chicago_dense.npz"
precomputed_chi = find_precomputed(CHI_DENSE_PATH)

if precomputed_chi:
    print(f"Chicago: reusing precomputed data from {precomputed_chi} -- skipping download.")
    _npz = np.load(precomputed_chi)
    chi_speed, chi_inflow, chi_demand = _npz["speed"], _npz["inflow"], _npz["demand"]
    if "d0" in _npz:
        chi_d0 = int(_npz["d0"])
    else:
        # Cached before the day-offset fix -- the leading empty days are already sliced out of
        # this array, so the true offset can't be recovered from it. Falling back to 0 would
        # silently mislabel every calendar feature if the real d0 wasn't 0, so warn loudly instead.
        chi_d0 = 0
        print("  WARNING: cached Chicago data has no saved d0 -- assuming day_offset=0. "
              "Delete chicago_dense.npz and rerun if calendar features look wrong.")
else:
    # Re-running this cell after a network failure re-downloads all 24 months from scratch unless we
    # checkpoint progress -- save the accumulator to disk after every month, and skip months already
    # done on the next run. Only helps within the same Kaggle session/container (a brand-new "Save &
    # Run All" commit starts with an empty disk), but that's exactly the case that matters when you're
    # re-running the same interactive session after a timeout.
    import pickle

    CHI_CKPT_PATH = "chicago_acc_checkpoint.pkl"

    if os.path.exists(CHI_CKPT_PATH):
        with open(CHI_CKPT_PATH, "rb") as f:
            chicago_acc, done_months = pickle.load(f)
        print(f"Resuming Chicago download -- {len(done_months)} month(s) already fetched: {sorted(done_months)}")
    else:
        chicago_acc = new_accumulators()
        done_months = set()

    # Chicago's feed identifies a tract by its centroid coordinate rather than by id, so each
    # distinct coordinate is matched to the tract polygon containing it (597/598 of the points
    # present in the data match, covering 99.94% of trips).
    import geopandas as _gpd
    from shapely.geometry import Point as _Point

    CHI_TRACTS_URL = "https://data.cityofchicago.org/resource/74p9-q2aq.geojson?$limit=5000"
    print("Downloading Chicago census-tract boundaries (2010, the vintage the taxi feed uses)...")
    _tr = requests.get(CHI_TRACTS_URL, timeout=300)
    _tr.raise_for_status()
    with open("chicago_tracts.geojson", "wb") as _f:
        _f.write(_tr.content)
    _tracts = _gpd.read_file("chicago_tracts.geojson")
    CHI_TRACT_W = zone_area_matrix(list(_tracts.geometry.values), lat_edges, lon_edges)
    print(f"  {len(_tracts)} tracts; cells reachable after spreading: {(CHI_TRACT_W.sum(axis=0) > 0).sum()}")

    _chi_pt_cache = {}

    def chi_tract_index(lats, lons):
        """Index of the tract containing each redacted point (-1 if none). Cached per distinct
        point -- there are only a few hundred of them in the whole dataset."""
        out = np.empty(len(lats), dtype=np.int64)
        todo = []
        keys = list(zip(np.round(lats, 6), np.round(lons, 6)))
        for i, k in enumerate(keys):
            hit = _chi_pt_cache.get(k)
            if hit is None:
                todo.append((i, k))
                out[i] = -1
            else:
                out[i] = hit
        if todo:
            pts = _gpd.GeoDataFrame(geometry=[_Point(k[1], k[0]) for _, k in todo], crs="EPSG:4326")
            j = _gpd.sjoin(pts, _tracts[["geometry"]], how="left", predicate="within")
            j = j[~j.index.duplicated(keep="first")]
            for (i, k), ti in zip(todo, j["index_right"].values):
                v = -1 if pd.isna(ti) else int(ti)
                _chi_pt_cache[k] = v
                out[i] = v
        return out

    for year in CHI_YEARS:
        for m in range(1, 13):
            if (year, m) in done_months:
                continue
            print(f"Chicago: fetching {year}-{m:02d} ...")
            month_df = fetch_chicago_month(year, m)
            accumulate_chicago(month_df, chicago_acc)
            del month_df
            done_months.add((year, m))
            with open(CHI_CKPT_PATH, "wb") as f:
                pickle.dump((chicago_acc, done_months), f)

    chi_demand = chicago_acc["pickup"] + chicago_acc["dropoff"]
    chi_inflow = chicago_acc["dropoff"]
    chi_speed = np.divide(chicago_acc["speed_sum"], chicago_acc["speed_n"],
                           out=np.zeros_like(chicago_acc["speed_sum"]), where=chicago_acc["speed_n"] > 0)

    (chi_speed, chi_inflow, chi_demand), chi_d0, d1 = trim_to_used_days(chi_speed, chi_inflow, chi_demand)
    print(f"Chicago: {chi_demand.shape[0]} days of data assembled (day offsets {chi_d0}-{d1} from {GLOBAL_START.date()}).")
    np.savez(CHI_DENSE_PATH, speed=chi_speed, inflow=chi_inflow, demand=chi_demand, d0=chi_d0,
             lat_edges=lat_edges, lon_edges=lon_edges)

write_city_csvs("chicago", chi_speed, chi_inflow, chi_demand)

### B.2 â€” NYC, 2024 through however much of 2026 TLC has published (zone-ID data, joined to official zone centroids)

In [ ]:
# NYC TLC yellow taxi trip records: hosted as monthly parquet files. TLC's official page is
# https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page -- if the URL pattern below has
# moved by the time you run this, get the current monthly links from that page instead.
# Confirmed (2026-09-13) against that page: all of 2024, all of 2025, and Jan-May 2026 are published,
# with a new month added roughly every 2 months -- so we just try every month up through the current
# one and let missing/not-yet-released months fail gracefully (caught below).
NYC_URL_TMPL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month:02d}.parquet"
NYC_ZONES_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip"  # official TLC zone shapefile
NYC_YEAR_MONTHS = [(y, m) for y in (2024, 2025) for m in range(1, 13)] + [(2026, m) for m in range(1, 13)]

NYC_DENSE_PATH = "nyc_dense.npz"

!pip -q install geopandas shapely pyogrio

import geopandas as gpd

# The zone shapefile is only needed to compute nyc_lat_edges/nyc_lon_edges -- skip downloading and
# parsing it entirely if a cached nyc_dense.npz (which now also stores those edges) is available.
_nyc_bbox_cache = find_precomputed(NYC_DENSE_PATH)
_nyc_cache_has_bbox = _nyc_bbox_cache and "lat_edges" in np.load(_nyc_bbox_cache)
if _nyc_cache_has_bbox:
    _npz_bbox = np.load(_nyc_bbox_cache)
    nyc_lat_edges, nyc_lon_edges = _npz_bbox["lat_edges"], _npz_bbox["lon_edges"]
    print(f"NYC bbox from cache ({_nyc_bbox_cache}): "
          f"lat[{nyc_lat_edges[0]:.4f},{nyc_lat_edges[-1]:.4f}] lon[{nyc_lon_edges[0]:.4f},{nyc_lon_edges[-1]:.4f}]")
else:
    try:
        r = requests.get(NYC_ZONES_URL, timeout=120)
        r.raise_for_status()
        with open("taxi_zones.zip", "wb") as f:
            f.write(r.content)
        # The zip's actual shapefile sits inside a "taxi_zones/" subfolder, not at the zip's top
        # level -- geopandas' "zip://taxi_zones.zip" shortcut only finds a file sitting right at the
        # top, so it fails to even recognize the zip's format. Extract for real and point straight
        # at the .shp file instead -- works regardless of the internal folder layout.
        import zipfile, glob
        with zipfile.ZipFile("taxi_zones.zip") as zf:
            zf.extractall("taxi_zones_extracted")
        shp_path = glob.glob("taxi_zones_extracted/**/*.shp", recursive=True)[0]
        zones = gpd.read_file(shp_path)
        zones_projected = zones.to_crs(epsg=3857)
        centroids = zones_projected.geometry.centroid.to_crs(epsg=4326)
        zones = zones.to_crs(epsg=4326)
        zones["lon"] = centroids.x
        zones["lat"] = centroids.y
        zone_centroids = zones.set_index("LocationID")[["lat", "lon"]]
        nyc_zone_geoms = zones.set_index("LocationID")["geometry"]   # needed for area spreading
        print(f"Loaded {len(zone_centroids)} NYC taxi zone centroids.")
    except Exception as e:
        raise RuntimeError(
            "Could not download/parse the NYC taxi zone shapefile from the URL above. "
            "Get the current link from https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page "
            f"and set NYC_ZONES_URL manually. Original error: {e}"
        )

    nyc_lat_min, nyc_lat_max = zone_centroids["lat"].min(), zone_centroids["lat"].max()
    nyc_lon_min, nyc_lon_max = zone_centroids["lon"].min(), zone_centroids["lon"].max()
    nyc_lat_edges, nyc_lon_edges = fine_grid_edges(nyc_lat_min, nyc_lat_max, nyc_lon_min, nyc_lon_max)
    print(f"NYC bbox from zone centroids: lat[{nyc_lat_min:.4f},{nyc_lat_max:.4f}] lon[{nyc_lon_min:.4f},{nyc_lon_max:.4f}]")

In [ ]:
def accumulate_nyc(df, acc):
    if df.empty:
        return
    df = df.dropna(subset=["tpep_pickup_datetime", "PULocationID", "DOLocationID"])
    df = df[df["PULocationID"].isin(zone_centroids.index) & df["DOLocationID"].isin(zone_centroids.index)]
    if df.empty:
        return
    ts = pd.to_datetime(df["tpep_pickup_datetime"])
    ts_end = pd.to_datetime(df["tpep_dropoff_datetime"])
    day = day_index(ts)
    hour = ts.dt.hour.values
    valid = (hour < TIME_SLOTS_PER_DAY) & (day >= 0) & (day < MAX_DAYS)

    # A trip records only WHICH ZONE it started in, never where inside it, so its demand is
    # distributed across that zone's real footprint.
    pu_idx = df["PULocationID"].map(NYC_ZONE_POS).values.astype(int)
    do_idx = df["DOLocationID"].map(NYC_ZONE_POS).values.astype(int)

    miles = df["trip_distance"].values
    secs = (ts_end - ts).dt.total_seconds().values
    speed = np.where(secs > 0, miles / (secs / 3600.0), np.nan)

    ones = np.ones(int(valid.sum()), dtype=np.float64)
    spread_into(acc["pickup"], day[valid], hour[valid], pu_idx[valid], ones, NYC_ZONE_W)
    spread_into(acc["dropoff"], day[valid], hour[valid], do_idx[valid], ones, NYC_ZONE_W)
    ok_speed = valid & np.isfinite(speed) & (speed > 0) & (speed < 100)
    spread_into(acc["speed_sum"], day[ok_speed], hour[ok_speed], pu_idx[ok_speed],
                speed[ok_speed].astype(np.float64), NYC_ZONE_W)
    spread_into(acc["speed_n"], day[ok_speed], hour[ok_speed], pu_idx[ok_speed],
                np.ones(int(ok_speed.sum()), dtype=np.float64), NYC_ZONE_W)

# Same reuse trick as Chicago above -- skip the up-to-36-month parquet download entirely if a
# previous run's cached nyc_dense.npz is available (via an attached "Notebook Output Files" input).
NYC_DENSE_PATH = "nyc_dense.npz"
precomputed_nyc = find_precomputed(NYC_DENSE_PATH)

if precomputed_nyc:
    print(f"NYC: reusing precomputed data from {precomputed_nyc} -- skipping download.")
    _npz = np.load(precomputed_nyc)
    nyc_speed, nyc_inflow, nyc_demand = _npz["speed"], _npz["inflow"], _npz["demand"]
    if "d0" in _npz:
        nyc_d0 = int(_npz["d0"])
    else:
        nyc_d0 = 0
        print("  WARNING: cached NYC data has no saved d0 -- assuming day_offset=0. "
              "Delete nyc_dense.npz and rerun if calendar features look wrong.")
else:
    # Area-spreading setup: one row per taxi zone giving how its area splits across grid cells.
    NYC_ZONE_POS = {int(z): i for i, z in enumerate(zone_centroids.index)}
    print(f"Building NYC zone-area distribution matrix for {len(NYC_ZONE_POS)} zones...")
    NYC_ZONE_W = zone_area_matrix([nyc_zone_geoms.loc[z] for z in zone_centroids.index],
                                   nyc_lat_edges, nyc_lon_edges)
    print(f"  grid cells reachable: {(NYC_ZONE_W.sum(axis=0) > 0).sum()}")

    nyc_acc = new_accumulators()
    for year, m in NYC_YEAR_MONTHS:
        url = NYC_URL_TMPL.format(year=year, month=m)
        print(f"NYC: fetching {url} ...")
        try:
            month_df = pd.read_parquet(url, columns=["tpep_pickup_datetime", "tpep_dropoff_datetime",
                                                       "trip_distance", "PULocationID", "DOLocationID"])
        except Exception as e:
            print(f"  Not available yet, skipping: {e}")
            continue
        accumulate_nyc(month_df, nyc_acc)
        del month_df

    nyc_demand = nyc_acc["pickup"] + nyc_acc["dropoff"]
    nyc_inflow = nyc_acc["dropoff"]
    nyc_speed = np.divide(nyc_acc["speed_sum"], nyc_acc["speed_n"],
                           out=np.zeros_like(nyc_acc["speed_sum"]), where=nyc_acc["speed_n"] > 0)

    (nyc_speed, nyc_inflow, nyc_demand), nyc_d0, d1 = trim_to_used_days(nyc_speed, nyc_inflow, nyc_demand)
    print(f"NYC: {nyc_demand.shape[0]} days of data assembled (day offsets {nyc_d0}-{d1} from {GLOBAL_START.date()}).")
    np.savez(NYC_DENSE_PATH, speed=nyc_speed, inflow=nyc_inflow, demand=nyc_demand, d0=nyc_d0,
             lat_edges=nyc_lat_edges, lon_edges=nyc_lon_edges)

write_city_csvs("nyc", nyc_speed, nyc_inflow, nyc_demand)

### B.2a â€” Porto, Portugal, Jul 2013â€“Jun 2014 (raw GPS polylines, single static archive, no zone-ID system)

A third, genuinely different city for the LOSO zero-shot test below, added alongside Chicago and
NYC. Unlike those two, Porto is **one static historical archive** (the UCI ECML/PKDD 2015 "Taxi
Service Trajectory Prediction Challenge"), not an ongoing feed -- downloaded once, then cached the
same way as `chicago_dense.npz`/`nyc_dense.npz`.

Trips give raw GPS, not a zone-ID system -- pickup is the first `[lon, lat]` point of each trip's
`POLYLINE`, dropoff is the last. Rows flagged `MISSING_DATA` or with an empty polyline are
dropped. Porto's real dates (2013â€“2014) are about a decade before this notebook's shared
`GLOBAL_START` (2024-01-01) -- reusing `day_index()` unmodified would produce negative day offsets
and silently drop every Porto trip, so Porto gets its own local day-index epoch for its own
accumulator array, then the trimmed offset is converted into a `GLOBAL_START`-relative number
(a large negative int, by design) so `calendar_row()` still reconstructs the correct real calendar
date. Porto also gets its own national holiday calendar (not the US one used for Chicago/NYC).

In [ ]:
# Porto, Portugal -- UCI ECML/PKDD 2015 "Taxi Service Trajectory Prediction Challenge".
# A single static archive (Jul 2013-Jun 2014), not an ongoing feed like Chicago/NYC above --
# downloaded once, then cached the same way via find_precomputed()/porto_dense.npz.
import time
import zipfile
import shutil

PORTO_ZIP_URL = "https://archive.ics.uci.edu/static/public/339/taxi+service+trajectory+prediction+challenge+ecml+pkdd+2015.zip"
PORTO_ZIP_PATH = "porto_taxi.zip"
PORTO_DENSE_PATH = "porto_dense.npz"

# Porto's real dates (2013-2014) are ~10 years before GLOBAL_START (2024-01-01) -- reusing
# day_index() unmodified would produce negative day offsets, fail accumulate_*'s "day >= 0"
# bound check, and silently drop every Porto trip. Porto gets its own LOCAL epoch for indexing
# into its own accumulator array (stays inside MAX_DAYS=1200), then the final trimmed offset is
# converted into a GLOBAL_START-relative number (a large negative int, by design) for CITY_D0/
# calendar_row() to use -- GLOBAL_START + timedelta(days=<large negative>) correctly reconstructs
# real 2013/2014 dates.
PORTO_LOCAL_START = datetime(2013, 7, 1)


def day_index_porto(ts_series):
    return (ts_series.dt.normalize() - pd.Timestamp(PORTO_LOCAL_START)).dt.days.values


def parse_first_last_point(polyline):
    """Pickup (first) and dropoff (last) [lon, lat] from a POLYLINE string, without a full JSON
    decode of every 15s-sampled point along the trip -- only the two endpoints are needed."""
    if not isinstance(polyline, str) or polyline == "[]" or len(polyline) < 5:
        return None
    inner = polyline[1:-1]
    first_end = inner.find(']')
    last_start, last_end = inner.rfind('['), inner.rfind(']')
    try:
        flon, flat = (float(v) for v in inner[1:first_end].split(','))
        llon, llat = (float(v) for v in inner[last_start + 1:last_end].split(','))
    except (ValueError, IndexError):
        return None
    return flon, flat, llon, llat


def download_with_retries(url, dest_path, attempts=4, timeout=600):
    for attempt in range(attempts):
        try:
            r = requests.get(url, stream=True, timeout=timeout)
            r.raise_for_status()
            with open(dest_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
            return
        except requests.exceptions.RequestException as e:
            if attempt == attempts - 1:
                raise
            wait = 5 * (2 ** attempt)
            print(f"  Porto zip download failed ({e}); retrying in {wait}s (attempt {attempt+1}/{attempts})...")
            time.sleep(wait)

In [ ]:
def accumulate_porto(df, acc, lat_edges, lon_edges):
    if df.empty:
        return
    df = df[df["MISSING_DATA"].astype(str).str.strip().str.lower() != "true"]
    df = df[df["POLYLINE"].notna() & (df["POLYLINE"] != "[]")]
    if df.empty:
        return
    parsed = df["POLYLINE"].map(parse_first_last_point)
    ok = parsed.notna()
    df, parsed = df[ok], parsed[ok]
    if df.empty:
        return
    pts = np.array(parsed.tolist(), dtype=np.float64)
    plon, plat, dlon, dlat = pts[:, 0], pts[:, 1], pts[:, 2], pts[:, 3]

    ts = pd.to_datetime(df["TIMESTAMP"], unit="s", utc=True).dt.tz_convert("Europe/Lisbon").dt.tz_localize(None)
    day = day_index_porto(ts)
    hour = ts.dt.hour.values
    valid = (hour < TIME_SLOTS_PER_DAY) & (day >= 0) & (day < MAX_DAYS)
    valid &= (plat >= lat_edges[0]) & (plat <= lat_edges[-1]) & (plon >= lon_edges[0]) & (plon <= lon_edges[-1])
    valid &= (dlat >= lat_edges[0]) & (dlat <= lat_edges[-1]) & (dlon >= lon_edges[0]) & (dlon <= lon_edges[-1])

    prow, pcol = assign_cell(plat[valid], plon[valid], lat_edges, lon_edges)
    drow, dcol = assign_cell(dlat[valid], dlon[valid], lat_edges, lon_edges)
    np.add.at(acc["pickup"], (day[valid], hour[valid], prow, pcol), 1)
    np.add.at(acc["dropoff"], (day[valid], hour[valid], drow, dcol), 1)


precomputed_porto = find_precomputed(PORTO_DENSE_PATH)

if precomputed_porto:
    print(f"Porto: reusing precomputed data from {precomputed_porto} -- skipping download.")
    _npz = np.load(precomputed_porto)
    porto_speed, porto_inflow, porto_demand = _npz["speed"], _npz["inflow"], _npz["demand"]
    porto_lat_edges, porto_lon_edges = _npz["lat_edges"], _npz["lon_edges"]
    if "d0" in _npz:
        porto_d0 = int(_npz["d0"])
    else:
        porto_d0 = (PORTO_LOCAL_START - GLOBAL_START).days
        print("  WARNING: cached Porto data has no saved d0 -- assuming no trimming. "
              "Delete porto_dense.npz and rerun if calendar features look wrong.")
else:
    if not os.path.exists(PORTO_ZIP_PATH):
        print("Porto: downloading UCI ECML/PKDD 2015 taxi-trajectory archive (~534MB, one-time)...")
        download_with_retries(PORTO_ZIP_URL, PORTO_ZIP_PATH)

    # The UCI archive nests the real data one level deeper: the outer zip's "train.csv.zip"
    # member is itself a zip containing train.csv (confirmed by directly inspecting the archive --
    # there is no top-level "train.csv"). Materialize that inner zip to disk once, then open it
    # normally so pandas gets proper random-access chunked reading.
    PORTO_INNER_ZIP_PATH = "porto_train_inner.zip"
    with zipfile.ZipFile(PORTO_ZIP_PATH) as outer_zf:
        outer_member = next(n for n in outer_zf.namelist() if n.lower().endswith("train.csv.zip"))
        if not os.path.exists(PORTO_INNER_ZIP_PATH):
            with outer_zf.open(outer_member) as src, open(PORTO_INNER_ZIP_PATH, "wb") as dst:
                shutil.copyfileobj(src, dst, length=1 << 20)

    with zipfile.ZipFile(PORTO_INNER_ZIP_PATH) as zf:
        names = zf.namelist()
        train_name = next(n for n in names if n.lower().endswith("train.csv"))

        # Bounding box from a sample of parsed pickup points, trimmed to the 0.5-99.5 percentile
        # so a handful of corrupt/outlier GPS fixes don't blow up the grid (Porto has no cheap
        # separate boundary fetch the way Chicago/NYC do, so it's derived from this same archive).
        print("Porto: sampling rows to establish bounding box...")
        with zf.open(train_name) as fh:
            sample = pd.read_csv(fh, usecols=["POLYLINE"], nrows=300_000)
        sample_pts = sample["POLYLINE"].map(parse_first_last_point).dropna()
        sample_arr = np.array(sample_pts.tolist(), dtype=np.float64)
        lat_lo, lat_hi = np.percentile(sample_arr[:, 1], [0.5, 99.5])
        lon_lo, lon_hi = np.percentile(sample_arr[:, 0], [0.5, 99.5])
        porto_lat_edges, porto_lon_edges = fine_grid_edges(lat_lo, lat_hi, lon_lo, lon_hi)
        print(f"Porto bbox from sampled data: lat[{lat_lo:.4f},{lat_hi:.4f}] lon[{lon_lo:.4f},{lon_hi:.4f}]")

        porto_acc = new_accumulators()
        print("Porto: streaming train.csv in chunks...")
        with zf.open(train_name) as fh:
            for chunk_i, chunk in enumerate(pd.read_csv(
                    fh, usecols=["TIMESTAMP", "MISSING_DATA", "POLYLINE"], chunksize=200_000)):
                accumulate_porto(chunk, porto_acc, porto_lat_edges, porto_lon_edges)
                if (chunk_i + 1) % 5 == 0:
                    print(f"  ...{(chunk_i + 1) * 200_000:,} rows processed")

    porto_demand = porto_acc["pickup"] + porto_acc["dropoff"]
    porto_inflow = porto_acc["dropoff"]
    porto_speed = np.zeros_like(porto_demand)  # no speed signal derived for Porto; unused downstream anyway

    (porto_demand, porto_inflow, porto_speed), porto_d0_local, porto_d1_local = trim_to_used_days(
        porto_demand, porto_inflow, porto_speed)
    porto_d0 = (PORTO_LOCAL_START - GLOBAL_START).days + porto_d0_local
    real_start = (PORTO_LOCAL_START + timedelta(days=int(porto_d0_local))).date()
    real_end = (PORTO_LOCAL_START + timedelta(days=int(porto_d1_local))).date()
    print(f"Porto: {porto_demand.shape[0]} days of data assembled ({real_start} to {real_end}); "
          f"porto_d0={porto_d0} (GLOBAL_START-relative, intentionally negative).")
    np.savez(PORTO_DENSE_PATH, speed=porto_speed, inflow=porto_inflow, demand=porto_demand,
             d0=porto_d0, lat_edges=porto_lat_edges, lon_edges=porto_lon_edges)

write_city_csvs("porto", porto_speed, porto_inflow, porto_demand)

### B.2b â€” Data-quality check on trip counts (before any clues are built)

Everything above (B.1/B.2) only did small, per-record filters while assembling the grid: drop
rows with a broken hour/day, drop Chicago rows with an impossible speed, drop NYC rows with a
missing pickup time or an unknown zone. None of that looks at the *aggregated* trip-count grid
itself. This cell checks three things on `chi_demand`/`nyc_demand` before Step 2 (static clues)
ever runs:

1. **Spikes/outliers in trip counts** â€” per cell, is any single hour wildly above that cell's own
   normal level (not just "busy", but a jump far outside its own typical range)?
2. **Gaps in the middle of the date range** â€” after trimming empty days off the very start/end,
   is there still a day (or hour) of zero trips *inside* the kept range, which would mean a real
   data gap rather than a genuinely quiet hour?
3. **Summary stats and plots** â€” basic numbers (mean/median/min/max/percentiles) plus a time-series
   plot and a histogram, so the shape of the data is actually visible somewhere.

This cell only reports/plots â€” it does not change `chi_demand`/`nyc_demand`, so it does not affect
any of the LOSO numbers below.

In [ ]:
import matplotlib.pyplot as plt

EDA_DIR = "urbanmind_results/eda"
os.makedirs(EDA_DIR, exist_ok=True)

eda_summary = {}
for city, dense, city_d0 in [("chicago", chi_demand, chi_d0), ("nyc", nyc_demand, nyc_d0), ("porto", porto_demand, porto_d0)]:
    days, hours, R, C = dense.shape
    flat = dense.reshape(days, hours, -1)          # (days, hours, R*C)
    per_hour_total = flat.sum(axis=2)               # (days, hours) -- citywide trips that hour
    hourly_series = per_hour_total.reshape(-1)      # flattened citywide hourly series, chronological

    stats = {
        "days_covered": int(days),
        "mean_trips_per_hour_citywide": float(hourly_series.mean()),
        "median_trips_per_hour_citywide": float(np.median(hourly_series)),
        "std_trips_per_hour_citywide": float(hourly_series.std()),
        "min_trips_per_hour_citywide": float(hourly_series.min()),
        "max_trips_per_hour_citywide": float(hourly_series.max()),
        "p01_trips_per_hour_citywide": float(np.percentile(hourly_series, 1)),
        "p99_trips_per_hour_citywide": float(np.percentile(hourly_series, 99)),
        "zero_hours_citywide": int((hourly_series == 0).sum()),
    }

    # --- gaps in the middle: any FULL DAY of zero trips still inside the trimmed [d0, d1) range? ---
    daily_total = per_hour_total.sum(axis=1)         # (days,) trips per day
    zero_days_idx = np.where(daily_total == 0)[0]
    zero_dates = [(GLOBAL_START + timedelta(days=int(city_d0 + i))).date().isoformat() for i in zero_days_idx]
    stats["zero_demand_days_in_middle"] = zero_dates

    # --- spikes: per CELL, hours far above that cell's OWN normal level (not a globally busy
    # cell -- a naturally busy downtown cell isn't a "spike", a cell suddenly firing many times
    # above its own typical value is). Only checked for cells with real average activity, so a
    # near-always-zero cell doesn't get flagged over one stray trip.
    cell_flat = dense.reshape(days * hours, -1)      # (day*hour, R*C)
    cell_mean = cell_flat.mean(axis=0)
    cell_std = cell_flat.std(axis=0)
    active = cell_mean > 0.5
    spike_threshold = cell_mean + 8 * cell_std       # 8 sigma above the cell's own mean -- rare-event bar
    spike_mask = (cell_flat > spike_threshold[None, :]) & active[None, :]
    spike_hits = np.argwhere(spike_mask)
    stats["spike_events"] = int(len(spike_hits))
    stats["cells_with_at_least_one_spike"] = int(len(np.unique(spike_hits[:, 1]))) if len(spike_hits) else 0
    stats["cells_checked_active"] = int(active.sum())

    eda_summary[city] = stats
    print(f"\n=== {city.upper()} trip-count EDA ===")
    for k, v in stats.items():
        if k == "zero_demand_days_in_middle":
            print(f"  {k}: {len(v)} day(s)" + (f" -> {v[:10]}" if v else ""))
        else:
            print(f"  {k}: {v}")

    fig, axes = plt.subplots(2, 1, figsize=(11, 6))
    axes[0].plot(hourly_series, linewidth=0.6, color="steelblue")
    axes[0].set_title(f"{city}: citywide trips per hour, whole period")
    axes[0].set_xlabel("hour index (chronological)")
    axes[0].set_ylabel("trips")

    nonzero = hourly_series[hourly_series > 0]
    axes[1].hist(np.log1p(nonzero), bins=60, color="steelblue")
    axes[1].set_title(f"{city}: distribution of nonzero hourly trip counts (log1p scale)")
    axes[1].set_xlabel("log1p(trips per hour)")
    axes[1].set_ylabel("count of hours")
    fig.tight_layout()
    out_path = f"{EDA_DIR}/{city}_trip_count_eda.png"
    fig.savefig(out_path, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"Saved {out_path}")

print("\neda_summary holds the per-city stats above for later reference.")
print("Note: this cell is report-only -- chi_demand/nyc_demand are unchanged, so it does not")
print("affect any LOSO result below.")

### B.2c â€” Fix what B.2b found

Two real issues came out of the check above:
- **NYC has a 24-day hole in the middle of the range** (2026-06-02 to 2026-06-25 â€” NYC TLC likely
  hadn't published that month yet when this ran). Left alone, the model would learn "NYC goes
  completely silent for 24 days", which is a publishing gap, not real demand.
- **Both cities have per-cell spikes** â€” hours far above what that specific cell normally does
  (Chicago: 272 events in 27 cells; NYC: 670 events in 109 cells).

Fixes, applied directly to `chi_demand`/`nyc_demand` so everything from Step 2 onward uses the
cleaned arrays:
- **Gap fill**: each missing day is replaced with the nearest PAST day that falls on the same
  weekday (stepping back 7, 14, 21, ... days until a non-missing one is found). Only ever looks
  backward, so this can't leak anything from the future into training or eval.
- **Spike cap**: any cell-hour flagged by the same rule as B.2b (more than 8 standard deviations
  above that cell's own mean, only checked on cells with real activity) is capped down to that
  threshold instead of left at its extreme raw value. Nothing is deleted, just capped.

In [ ]:
def cap_spikes(dense):
    days, hours, R, C = dense.shape
    flat = dense.reshape(days * hours, -1)
    cell_mean = flat.mean(axis=0)
    cell_std = flat.std(axis=0)
    active = cell_mean > 0.5
    threshold = cell_mean + 8 * cell_std
    spike_mask = (flat > threshold[None, :]) & active[None, :]
    n_capped = int(spike_mask.sum())
    flat = np.where(spike_mask, np.broadcast_to(threshold, flat.shape), flat)
    return flat.reshape(days, hours, R, C), n_capped

def fill_middle_gaps(dense):
    days = dense.shape[0]
    daily_total = dense.reshape(days, -1).sum(axis=1)
    missing_days = np.where(daily_total == 0)[0]
    filled = dense.copy()
    n_filled = 0
    for d in missing_days:
        step = 7
        while True:
            src = d - step
            if src < 0:
                break  # no past data at all -- shouldn't happen, trim_to_used_days already
                       # removed any leading all-zero days
            if daily_total[src] > 0:
                filled[d] = filled[src]
                n_filled += 1
                break
            step += 7
    return filled, n_filled

def quick_check(dense):
    days = dense.shape[0]
    remaining_gap_days = int((dense.reshape(days, -1).sum(axis=1) == 0).sum())
    flat = dense.reshape(-1, dense.shape[2] * dense.shape[3])
    cell_mean, cell_std = flat.mean(axis=0), flat.std(axis=0)
    active = cell_mean > 0.5
    threshold = cell_mean + 8 * cell_std
    remaining_spikes = int(((flat > threshold[None, :]) & active[None, :]).sum())
    return remaining_gap_days, remaining_spikes

print("Cleaning Chicago trip counts...")
chi_demand, chi_days_filled = fill_middle_gaps(chi_demand)
chi_demand, chi_spikes_capped = cap_spikes(chi_demand)
print(f"  middle-gap days filled: {chi_days_filled} | spikes capped: {chi_spikes_capped}")

print("Cleaning NYC trip counts...")
nyc_demand, nyc_days_filled = fill_middle_gaps(nyc_demand)
nyc_demand, nyc_spikes_capped = cap_spikes(nyc_demand)
print(f"  middle-gap days filled: {nyc_days_filled} | spikes capped: {nyc_spikes_capped}")

print("Cleaning Porto trip counts...")
porto_demand, porto_days_filled = fill_middle_gaps(porto_demand)
porto_demand, porto_spikes_capped = cap_spikes(porto_demand)
print(f"  middle-gap days filled: {porto_days_filled} | spikes capped: {porto_spikes_capped}")

chi_gap_check, chi_spike_check = quick_check(chi_demand)
nyc_gap_check, nyc_spike_check = quick_check(nyc_demand)
porto_gap_check, porto_spike_check = quick_check(porto_demand)
print(f"\nVerify after cleaning -- chicago: {chi_gap_check} gap day(s), {chi_spike_check} spike(s) remaining")
print(f"Verify after cleaning -- nyc: {nyc_gap_check} gap day(s), {nyc_spike_check} spike(s) remaining")
print(f"Verify after cleaning -- porto: {porto_gap_check} gap day(s), {porto_spike_check} spike(s) remaining")
print("\nchi_demand/nyc_demand/porto_demand updated in place -- Step 2 (static clues) onward now sees the cleaned data.")

### B.8 â€” Type II zero-shot: MAE first, then XGBoost, tested both directions (LOSO)

**What this tests:** can we guess a brand-new city's demand pattern using **zero** real demand history from it â€” only facts about the map and the calendar? And does an MAE (masked-reconstruction) step actually help, or is it dead weight over plain XGBoost?

**The rule, unchanged and strict:** whichever city we're testing on, its real demand is read exactly once â€” at the very end, only to grade the guess. Never during training, never as a model input.

**The method, in order:**
1. **MAE step** â€” take the source city's place-type + location clues (never demand), hide some of them, train a small model to guess the hidden ones back. Zero demand involved. This produces a learned "sense of what this spot is like" for every cell.
2. **XGBoost step** â€” trained only on the source city: `[that learned sense + calendar] -> real demand share`. Target city still not involved.
3. **Apply both, frozen, to the target city** â€” feed the target's own place-type/location clues through the same frozen MAE model, then the same frozen XGBoost, to get a guess. Only now do we look at the target's real numbers, purely to check the guess.

**Runs all 6 directed pairs across three cities (LOSO):** Chicagoâ†”NYC, Chicagoâ†”Porto, and NYCâ†”Porto, each trained zero-shot in both directions -- 6 train/test combinations total.

**Clues used â€” 16 static (never change over time) + 9 calendar (change every hour):**
- Place-type (8, independent â€” a spot can score on several at once): food, retail, office, transit, education, health, leisure, civic
- Location (5, all computed automatically from lat/long, none of them raw grid position, distances relative to that city's own bounding-box diagonal so cities of different sizes are comparable): distance to downtown (real city center, looked up automatically â€” not hardcoded), distance to nearest transit stop, distance to water, overall busyness (density of the 8 place-type counts), mix score (how varied the place-types are)
- Data-quality flags (3): whether the downtown/transit/water map query returned nothing for that city, so a failed lookup is never disguised as a real distance
- Calendar (9): hour of day, day of week, weekend flag, holiday flag (US federal for Chicago/NYC, Portugal's own calendar for Porto), month, rush-hour flag

**Compared against, for each direction:**
- Guessing evenly across all cells
- Copying the source city's hour-of-day pattern straight onto the target (no learning)
- XGBoost alone, skipping the MAE step (checks whether MAE is actually contributing anything)
- MAE + XGBoost (the actual method)

**Metrics:** MAE, RMSE, and **sMAPE** (not plain MAPE â€” with mostly-tiny share values across 2,000 cells, plain MAPE explodes; sMAPE is bounded and matches the metric already used in this user's prior LSTM paper), plus JSD and spatial Spearman correlation as extra context (do the busy areas line up, regardless of the exact numbers).

In [ ]:
# --- B.8 setup: comparable static clues, computed from a lat/lon bounding box only ---
# V23 feature contract: POI counts are densities (per kmÂ²); distances are divided by each
# city's bounding-box diagonal. These transformations use map geometry only, never target demand.

!pip -q install osmnx xgboost

import osmnx as ox
import xgboost as xgb
from scipy.stats import spearmanr, rankdata
from pandas.tseries.holiday import USFederalHolidayCalendar

POI_GROUPS = {
    "food": {"amenity": ["restaurant", "cafe", "bar", "fast_food", "pub"]},
    "retail": {"shop": True},
    "office": {"office": True},
    "transit": {"railway": ["station", "subway_entrance", "tram_stop"], "highway": ["bus_stop"]},
    "education": {"amenity": ["school", "university", "college", "library"]},
    "health": {"amenity": ["hospital", "clinic", "pharmacy", "doctors"]},
    "leisure": {"leisure": True, "tourism": True},
    "civic": {"amenity": ["bank", "townhall", "courthouse", "post_office"]},
}
DOWNTOWN_QUERY = {"chicago": "Chicago Loop, Chicago, IL, USA", "nyc": "Midtown Manhattan, New York, NY, USA", "porto": "Porto, Portugal"}


def _features_from_bbox(north, south, east, west, tags):
    errors = []
    for call in (
        lambda: ox.features_from_bbox(bbox=(west, south, east, north), tags=tags),  # OSMnx >= 2
        lambda: ox.features_from_bbox(north, south, east, west, tags),              # OSMnx 1.x
    ):
        try:
            return call()
        except Exception as exc:
            errors.append(f"{type(exc).__name__}: {exc}")
    raise RuntimeError(" | ".join(errors))


def fetch_points(north, south, east, west, tags, label):
    """Return representative WGS84 (lat, lon) points; fail visibly but safely for a POI class."""
    try:
        gdf = _features_from_bbox(north, south, east, west, tags)
        if gdf is None or len(gdf) == 0:
            print(f"  {label}: 0 features")
            return np.empty((0, 2), dtype=np.float32)
        if gdf.crs is None:
            gdf = gdf.set_crs(epsg=4326)
        points = gdf.to_crs(epsg=3857).geometry.representative_point().to_crs(epsg=4326)
        out = np.stack([points.y.values, points.x.values], axis=1).astype(np.float32)
        print(f"  {label}: {len(out)} features")
        return out
    except Exception as exc:
        print(f"  {label}: FAILED ({type(exc).__name__}: {exc}) -- recorded as missing")
        return np.empty((0, 2), dtype=np.float32)


def cell_centers(lat_edges, lon_edges):
    lat_c = (lat_edges[:-1] + lat_edges[1:]) / 2
    lon_c = (lon_edges[:-1] + lon_edges[1:]) / 2
    return np.meshgrid(lat_c, lon_c, indexing="ij")


def haversine_km(lat1, lon1, lat2, lon2):
    radius = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dlambda = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlambda / 2) ** 2
    return 2 * radius * np.arcsin(np.sqrt(a))


def grid_geometry(lat_edges, lon_edges):
    south, north = float(lat_edges[0]), float(lat_edges[-1])
    west, east = float(lon_edges[0]), float(lon_edges[-1])
    mid_lat, mid_lon = (south + north) / 2, (west + east) / 2
    height = haversine_km(south, mid_lon, north, mid_lon)
    width = haversine_km(mid_lat, west, mid_lat, east)
    cell_area = max((height / GRID_ROWS) * (width / GRID_COLS), 1e-6)
    diagonal = max(haversine_km(south, west, north, east), 1e-6)
    return cell_area, diagonal


def nearest_distance(cell_lat, cell_lon, points, fallback_km):
    """Distance plus a missingness flag; a failed map query is never disguised as a real distance."""
    if len(points) == 0:
        return np.full(cell_lat.shape, fallback_km, dtype=np.float32), np.ones(cell_lat.shape, dtype=np.float32)
    distances = np.stack([haversine_km(cell_lat, cell_lon, lat, lon) for lat, lon in points], axis=0)
    return distances.min(axis=0).astype(np.float32), np.zeros(cell_lat.shape, dtype=np.float32)


def build_static_features(city_name, lat_edges, lon_edges):
    north, south = float(lat_edges[-1]), float(lat_edges[0])
    east, west = float(lon_edges[-1]), float(lon_edges[0])
    lat_c, lon_c = cell_centers(lat_edges, lon_edges)
    cell_area_km2, city_diagonal_km = grid_geometry(lat_edges, lon_edges)

    poi_counts = np.zeros((GRID_ROWS, GRID_COLS, len(POI_GROUPS)), dtype=np.float32)
    poi_points = {}
    for column, (name, tags) in enumerate(POI_GROUPS.items()):
        points = fetch_points(north, south, east, west, tags, f"{city_name}/{name}")
        poi_points[name] = points
        if len(points):
            rows, cols = assign_cell(points[:, 0], points[:, 1], lat_edges, lon_edges)
            np.add.at(poi_counts[:, :, column], (rows, cols), 1)

    water_points = fetch_points(north, south, east, west, {"natural": "water"}, f"{city_name}/water")
    try:
        downtown_lat, downtown_lon = ox.geocode(DOWNTOWN_QUERY[city_name])
        downtown_points = np.array([[downtown_lat, downtown_lon]], dtype=np.float32)
        print(f"  {city_name}/downtown: geocoded to ({downtown_lat:.4f}, {downtown_lon:.4f})")
    except Exception as exc:
        downtown_points = np.array([[float(lat_c.mean()), float(lon_c.mean())]], dtype=np.float32)
        print(f"  {city_name}/downtown: geocoding FAILED ({exc}) -- recorded as bbox-centre fallback")

    # Airport was removed in V23: the V22 NYC query failed and its synthetic constant caused
    # catastrophic cross-city scaling. It can return only after a verified common source is added.
    dist_downtown, downtown_missing = nearest_distance(lat_c, lon_c, downtown_points, city_diagonal_km / 2)
    dist_transit, transit_missing = nearest_distance(lat_c, lon_c, poi_points["transit"], city_diagonal_km / 2)
    dist_water, water_missing = nearest_distance(lat_c, lon_c, water_points, city_diagonal_km / 2)

    poi_density = poi_counts / cell_area_km2
    busyness = poi_density.sum(axis=-1)
    props = poi_density / np.clip(busyness[..., None], 1e-9, None)
    mix_score = -np.sum(np.where(props > 0, props * np.log(props + 1e-9), 0.0), axis=-1)

    static = np.concatenate([
        np.log1p(poi_density),
        np.log1p(dist_downtown / city_diagonal_km)[..., None],
        np.log1p(dist_transit / city_diagonal_km)[..., None],
        np.log1p(dist_water / city_diagonal_km)[..., None],
        np.log1p(busyness)[..., None], mix_score[..., None],
        downtown_missing[..., None], transit_missing[..., None], water_missing[..., None],
    ], axis=-1)
    print(f"  {city_name}: cell area={cell_area_km2:.3f} kmÂ², diagonal={city_diagonal_km:.2f} km")
    return static.astype(np.float32)


# Querying OSM for 8 POI categories + water + downtown geocoding, per city, is slow and another
# network dependency worth skipping on repeat runs -- these clues depend only on each city's fixed
# bounding box (never on demand data), so they're cached exactly like the dense demand arrays above.
STATIC_CORE_PATH = "static_core.npz"
_static_core_cache = find_precomputed(STATIC_CORE_PATH)
if _static_core_cache:
    _npz_static = np.load(_static_core_cache)
    STATIC_CHI = _npz_static["STATIC_CHI"]
    STATIC_NYC = _npz_static["STATIC_NYC"]
    STATIC_PORTO = _npz_static["STATIC_PORTO"]
    print(f"Static clues: reusing cached {_static_core_cache} -- skipping OSM queries for chicago/nyc/porto.")
else:
    print("Building Chicago static clues (map-only, normalized units)...")
    STATIC_CHI = build_static_features("chicago", lat_edges, lon_edges)
    print("Building NYC static clues (map-only, normalized units)...")
    STATIC_NYC = build_static_features("nyc", nyc_lat_edges, nyc_lon_edges)
    print("Building Porto static clues (map-only, normalized units)...")
    STATIC_PORTO = build_static_features("porto", porto_lat_edges, porto_lon_edges)
    np.savez(STATIC_CORE_PATH, STATIC_CHI=STATIC_CHI, STATIC_NYC=STATIC_NYC, STATIC_PORTO=STATIC_PORTO)

N_STATIC = STATIC_CHI.shape[-1]
assert N_STATIC == STATIC_NYC.shape[-1] == STATIC_PORTO.shape[-1]
print(f"Static clue columns per cell: {N_STATIC} (8 POI density + 5 location/mix + 3 quality flags)")


In [ ]:
# --- B.8 calendar clues (computed automatically from a date + hour) ---
US_HOLIDAYS = set(USFederalHolidayCalendar().holidays(start="2023-01-01", end="2027-01-01").date)

!pip -q install holidays
import holidays as _holidays_pkg
# Porto's real dates fall in 2013-2014 (see PORTO_LOCAL_START below) -- its own national calendar,
# not the US one used for Chicago/NYC, since reusing US_HOLIDAYS would mislabel every Portuguese
# holiday as a regular day and vice versa.
PT_HOLIDAYS = set(_holidays_pkg.Portugal(years=range(2013, 2015)).keys())
CITY_HOLIDAYS = {"chicago": US_HOLIDAYS, "nyc": US_HOLIDAYS, "porto": PT_HOLIDAYS}

N_CAL = 9  # hour sin/cos, dow sin/cos, weekend, holiday, month sin/cos, rush-hour

def calendar_row(day_offset, hour, city="chicago"):
    d = (GLOBAL_START + timedelta(days=int(day_offset))).date()
    dow = d.weekday()
    is_rush = 1.0 if hour in (7, 8, 9, 16, 17, 18) else 0.0
    holiday_set = CITY_HOLIDAYS.get(city, US_HOLIDAYS)
    return np.array([
        np.sin(2 * np.pi * hour / 24), np.cos(2 * np.pi * hour / 24),
        np.sin(2 * np.pi * dow / 7), np.cos(2 * np.pi * dow / 7),
        1.0 if dow >= 5 else 0.0,
        1.0 if d in holiday_set else 0.0,
        np.sin(2 * np.pi * d.month / 12), np.cos(2 * np.pi * d.month / 12),
        is_rush,
    ], dtype=np.float32)

# --- MAE step: hide some static clues per cell, learn to reconstruct them. Zero demand used. ---
class ClueMAE(nn.Module):
    def __init__(self, n_static, embed_dim=8, hidden=48):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.LayerNorm(n_static), nn.Linear(n_static, hidden), nn.GELU(),
            nn.Dropout(0.10), nn.Linear(hidden, embed_dim),
        )
        self.decoder = nn.Sequential(nn.Linear(embed_dim, hidden), nn.GELU(), nn.Linear(hidden, n_static))

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

def train_clue_mae(static_grid, epochs=300, mask_ratio=0.3):
    """Masked reconstruction of source-city map clues only; target demand is never accessed."""
    torch.manual_seed(42)
    X = torch.tensor(static_grid.reshape(-1, static_grid.shape[-1]), device=device)
    model = ClueMAE(X.shape[1]).to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3)
    for ep in range(epochs):
        mask = (torch.rand_like(X) < mask_ratio)
        X_masked = X.clone()
        X_masked[mask] = 0.0
        opt.zero_grad()
        recon, _ = model(X_masked)
        loss = ((recon - X)[mask] ** 2).mean()
        loss.backward(); opt.step()
        if (ep + 1) % max(1, epochs // 5) == 0:
            print(f"  [MAE] epoch {ep+1}/{epochs}  reconstruction error on hidden clues = {loss.item():.5f}")
    model.eval()
    with torch.no_grad():
        _, Z = model(X)
    return model, Z.cpu().numpy().reshape(static_grid.shape[0], static_grid.shape[1], -1)

def apply_clue_mae(model, static_grid):
    X = torch.tensor(static_grid.reshape(-1, static_grid.shape[-1]), device=device)
    model.eval()
    with torch.no_grad():
        _, Z = model(X)
    return Z.cpu().numpy().reshape(static_grid.shape[0], static_grid.shape[1], -1)

def robust_source_standardize(source_grid, target_grid, clip=8.0):
    """Source-only median/IQR scaling with safe handling of constant source features.

    Target static geography is allowed in Type-II zero-shot; target demand is not. The transform
    itself is fitted only on source cells. Clipping stops an unseen map feature from pushing a
    tree/MAE far beyond its training support.
    """
    source = source_grid.reshape(-1, source_grid.shape[-1])
    q25, median, q75 = np.percentile(source, [25, 50, 75], axis=0)
    scale = q75 - q25
    constant = scale < 1e-5
    scale[constant] = 1.0
    src_scaled = np.clip((source_grid - median) / scale, -clip, clip).astype(np.float32)
    tgt_scaled = np.clip((target_grid - median) / scale, -clip, clip).astype(np.float32)
    return src_scaled, tgt_scaled, int(constant.sum())

def make_xgb():
    # Shallower, regularised trees reduce memorisation of one city's individual grid cells.
    return xgb.XGBRegressor(
        n_estimators=400, max_depth=4, learning_rate=0.05, min_child_weight=10,
        subsample=0.80, colsample_bytree=0.90, reg_lambda=5.0,
        objective="reg:squarederror", tree_method="hist", random_state=42,
    )

# --- Build (day, hour) -> full share-over-all-cells samples, for evaluation + climatology ---
def build_share_dataset(dense_demand, day_offset_start=0, city="chicago"):
    days, hours = dense_demand.shape[0], dense_demand.shape[1]
    flat = dense_demand.reshape(days, hours, -1)
    totals = flat.sum(axis=2)
    cal_list, share_list, hour_list, total_list, day_list = [], [], [], [], []
    for d in range(days):
        for h in range(hours):
            t = totals[d, h]
            if t <= 0:
                continue
            cal_list.append(calendar_row(day_offset_start + d, h, city=city))
            share_list.append(flat[d, h] / t)
            hour_list.append(h)
            total_list.append(t)  # real citywide total that hour -- needed to convert share back to trip counts
            day_list.append(day_offset_start + d)  # true day offset from GLOBAL_START, for date/weekday lookups
    return (np.stack(cal_list).astype(np.float32),
            np.stack(share_list).astype(np.float32),
            np.array(hour_list),
            np.array(total_list, dtype=np.float32),
            np.array(day_list))

def climatology_from(cal, share, hours, n_cells):
    """Source city's own mean share-per-hour-of-day map -- used as the 'just copy it' baseline."""
    clim = np.zeros((24, n_cells), dtype=np.float32)
    for h in range(24):
        m = hours == h
        clim[h] = share[m].mean(axis=0) if m.any() else (1.0 / n_cells)
    return clim

# --- XGBoost training rows: subsampled (day,hour) instances x all cells, source city only ---
def build_training_rows(dense_demand, feature_grid, day_offset_start=0, max_rows=2_000_000, seed=0, city="chicago"):
    days, hours, R, C = dense_demand.shape
    F = feature_grid.shape[-1]
    totals = dense_demand.reshape(days, hours, -1).sum(axis=2)
    feat_flat = feature_grid.reshape(-1, F)

    rng = np.random.default_rng(seed)
    pairs = [(d, h) for d in range(days) for h in range(hours) if totals[d, h] > 0]
    max_samples = max(1, max_rows // (R * C))
    if len(pairs) > max_samples:
        idx = rng.choice(len(pairs), size=max_samples, replace=False)
        pairs = [pairs[i] for i in idx]

    X_list, y_list = [], []
    for d, h in pairs:
        cal_tile = np.tile(calendar_row(day_offset_start + d, h, city=city), (R * C, 1))
        X_list.append(np.concatenate([feat_flat, cal_tile], axis=1))
        y_list.append((dense_demand[d, h].reshape(-1) / totals[d, h]).astype(np.float32))
    return np.concatenate(X_list, axis=0).astype(np.float32), np.concatenate(y_list, axis=0)

def predict_all(model, feature_grid, cal_vecs, n_cells):
    # Bug found by inspecting a real run: clipping negative predictions straight to zero means
    # that whenever the model extrapolates badly on an unfamiliar target city (every cell's raw
    # prediction comes out negative -- seen in practice when trained on one city, applied cold to
    # another), every clipped value becomes exactly 0, and the result silently collapses to the
    # flat uniform guess -- discarding whatever real relative signal the model still had. Shifting
    # by the batch minimum instead preserves the model's relative ranking between cells no matter
    # how far off the absolute scale is, and never needs a fallback (the sum is always positive).
    flat = feature_grid.reshape(-1, feature_grid.shape[-1])
    preds = []
    for cal in cal_vecs:
        Xr = np.concatenate([flat, np.tile(cal, (flat.shape[0], 1))], axis=1)
        raw = model.predict(Xr)
        shifted = raw - raw.min() + 1e-6
        preds.append(shifted / shifted.sum())
    return np.stack(preds, axis=0)

# --- Scoring: MAE, RMSE, sMAPE (primary) + JSD, spatial Spearman (extra context) ---
def smape(pred, true, eps=1e-9):
    # Bounded 0-100% (denominator is the plain sum, not the sum's average) -- matches the
    # convention the user's LSTM paper uses, so both papers report sMAPE on the same scale.
    return float(np.mean(np.abs(pred - true) / (np.abs(pred) + np.abs(true) + eps)) * 100)

def jsd(p, q, eps=1e-9):
    p = np.clip(p, eps, None); q = np.clip(q, eps, None)
    m = 0.5 * (p + q)
    kl = lambda a, b: (a * (np.log(a) - np.log(b))).sum(axis=-1)
    return float(np.mean(0.5 * kl(p, m) + 0.5 * kl(q, m)))

def count_errors(pred_share, true_share, totals):
    """Converts share predictions back into trip counts (share * that hour's real total trips)
    so MAE/RMSE come out in the same units most papers report (trips per cell per hour), instead
    of a fraction-of-citywide-total. NOTE: this uses the target city's own real total demand per
    hour to do the conversion -- that's genuine target-city data, so these two numbers alone are
    not a pure zero-shot result, just a same-units comparison view. The share-based numbers above
    remain the actual zero-shot result (zero target demand used anywhere)."""
    pred_counts = pred_share * totals[:, None]
    true_counts = true_share * totals[:, None]
    mae = float(np.mean(np.abs(pred_counts - true_counts)))
    rmse = float(np.sqrt(np.mean((pred_counts - true_counts) ** 2)))
    return mae, rmse

def score(pred, true, name, totals=None):
    mae = float(np.mean(np.abs(pred - true)))
    rmse = float(np.sqrt(np.mean((pred - true) ** 2)))
    sm = smape(pred, true)
    rho = float(np.mean([spearmanr(pred[i], true[i]).correlation
                         for i in range(0, len(true), max(1, len(true) // 200))]))
    d = {"MAE": mae, "RMSE": rmse, "sMAPE": sm, "JSD": jsd(pred, true), "spatial_spearman": rho}
    line = f"  {name:32s} MAE={mae:.6f}  RMSE={rmse:.6f}  sMAPE={sm:6.2f}%  JSD={d['JSD']:.5f}  spearman={rho:.4f}"
    if totals is not None:
        mae_t, rmse_t = count_errors(pred, true, totals)
        d["MAE_trips"], d["RMSE_trips"] = mae_t, rmse_t
        line += f"  |  trips: MAE={mae_t:.3f}  RMSE={rmse_t:.3f}"
    print(line)
    return d

def stratified_trip_errors(pred_share, true_share, totals, busy_frac=0.05):
    """Splits target cells into 'busy' (top busy_frac by real total demand across the eval
    window) and 'quiet' (the rest), and reports MAE_trips separately for each -- lets us see
    whether a fix actually helps the cells that matter instead of being hidden by the huge
    number of easy near-zero cells."""
    true_counts = true_share * totals[:, None]
    pred_counts = pred_share * totals[:, None]
    cell_totals = true_counts.mean(axis=0)
    order = np.argsort(-cell_totals)
    n_busy = max(1, int(len(order) * busy_frac))
    busy_cells, quiet_cells = order[:n_busy], order[n_busy:]
    def mae_for(cells):
        return float(np.mean(np.abs(pred_counts[:, cells] - true_counts[:, cells])))
    return {
        "busy_frac": busy_frac, "n_busy_cells": int(len(busy_cells)), "n_quiet_cells": int(len(quiet_cells)),
        "MAE_trips_busy": mae_for(busy_cells), "MAE_trips_quiet": mae_for(quiet_cells),
        "MAE_trips_all": float(np.mean(np.abs(pred_counts - true_counts))),
    }


def ranking_diagnostics(pred_share, true_share, totals, ks=(1, 5, 10, 20, 50)):
    """Does the model pick the cells that are actually busy? A separate question from whether it
    sizes them right -- and the one that turned out to be the real problem. Sharpening the
    predictions made busy-cell error worse, which is only possible if the picked cells are wrong.
    """
    true_counts = (true_share * totals[:, None]).mean(axis=0)
    pred_counts = (pred_share * totals[:, None]).mean(axis=0)
    to, po = np.argsort(-true_counts), np.argsort(-pred_counts)
    out = {f"top{k}_overlap": int(len(set(po[:k].tolist()) & set(to[:k].tolist()))) for k in ks}
    out["rank_of_true_busiest"] = int(np.where(po == to[0])[0][0]) + 1
    return out


print("B.8 functions ready.")

In [ ]:
# --- B.8 LOSO: train on one city, zero-shot test on the other. Both directions. ---

N_CELLS = GRID_ROWS * GRID_COLS
CITY_STATIC = {"chicago": STATIC_CHI, "nyc": STATIC_NYC, "porto": STATIC_PORTO}
CITY_DEMAND = {"chicago": chi_demand, "nyc": nyc_demand, "porto": porto_demand}
CITY_D0 = {"chicago": chi_d0, "nyc": nyc_d0, "porto": porto_d0}  # true day-offset of each city's day-index 0 (FIX: calendar alignment)

type2_results = {}
type2_raw = {}  # per-direction arrays kept around for the spatial/error/feature-importance analysis below

for src, tgt in [("chicago", "nyc"), ("nyc", "chicago"),
                  ("chicago", "porto"), ("porto", "chicago"),
                  ("nyc", "porto"), ("porto", "nyc")]:
    print(f"\n=== TYPE II LOSO: train on {src}, zero-shot test on {tgt} (zero {tgt} demand used in training) ===")
    try:
        src_static_raw, tgt_static_raw = CITY_STATIC[src], CITY_STATIC[tgt]
        src_static, tgt_static, n_constant = robust_source_standardize(src_static_raw, tgt_static_raw)
        print(f"  [diag] source-only robust scaling: {n_constant} constant source feature(s); clip=Â±8")
        print(f"  [diag] {src}_static range: [{src_static.min():.2f}, {src_static.max():.2f}] std={src_static.std():.2f}")
        print(f"  [diag] {tgt}_static range after source transform: [{tgt_static.min():.2f}, {tgt_static.max():.2f}] std={tgt_static.std():.2f}")

        # FIX: pass each city's true day-offset (CITY_D0) so calendar_row() computes the correct
        # calendar date -- previously this defaulted to day_offset_start=0 regardless of how many
        # leading empty days trim_to_used_days() had sliced off, silently mislabeling day-of-week/
        # weekend/holiday/month for every sample whenever a city's data didn't start exactly on
        # GLOBAL_START.
        cal_src, share_src, hours_src, totals_src, days_src = build_share_dataset(
            CITY_DEMAND[src], day_offset_start=CITY_D0[src], city=src)
        cal_tgt, share_tgt, hours_tgt, totals_tgt, days_tgt = build_share_dataset(
            CITY_DEMAND[tgt], day_offset_start=CITY_D0[tgt], city=tgt)
        print(f"{src} samples available: {len(cal_src)} | {tgt} eval samples: {len(cal_tgt)}")

        uniform_share = np.full(N_CELLS, 1.0 / N_CELLS, dtype=np.float32)
        clim = climatology_from(cal_src, share_src, hours_src, N_CELLS)

        print(f"Training MAE on {src}'s static clues only (no demand)...")
        mae_model, src_embed = train_clue_mae(src_static, epochs=300)
        tgt_embed = apply_clue_mae(mae_model, tgt_static)
        print(f"  [diag] {src}_embed range: [{src_embed.min():.3f}, {src_embed.max():.3f}] std={src_embed.std():.3f}")
        print(f"  [diag] {tgt}_embed range: [{tgt_embed.min():.3f}, {tgt_embed.max():.3f}] std={tgt_embed.std():.3f}")

        print(f"Training XGBoost (no MAE) on {src} only...")
        Xraw, yraw = build_training_rows(CITY_DEMAND[src], src_static, day_offset_start=CITY_D0[src], city=src)
        xgb_raw = make_xgb()
        xgb_raw.fit(Xraw, yraw)

        # The MAE is an augmentation, not a replacement: raw comparable geography is retained
        # alongside the learned embedding, so reconstruction cannot erase a useful source feature.
        src_mae_features = np.concatenate([src_static, src_embed], axis=-1)
        tgt_mae_features = np.concatenate([tgt_static, tgt_embed], axis=-1)
        print(f"Training XGBoost on raw clues + MAE embeddings from {src} only...")
        Xmae, ymae = build_training_rows(CITY_DEMAND[src], src_mae_features, day_offset_start=CITY_D0[src], city=src)
        xgb_mae = make_xgb()
        xgb_mae.fit(Xmae, ymae)

        pred_xgb_raw = predict_all(xgb_raw, tgt_static, cal_tgt, N_CELLS)
        pred_xgb_mae = predict_all(xgb_mae, tgt_mae_features, cal_tgt, N_CELLS)
        print(f"  [diag] pred_xgb_raw per-sample cell-std (mean over samples): {pred_xgb_raw.std(axis=1).mean():.8f}  (0 = collapsed to flat)")
        print(f"  [diag] pred_xgb_mae per-sample cell-std (mean over samples): {pred_xgb_mae.std(axis=1).mean():.8f}  (0 = collapsed to flat)")

        key = f"{src}_to_{tgt}"
        print(f"\n-- Results: {key} --")
        # totals=totals_tgt below also reports MAE/RMSE in trip counts (same units as most papers) --
        # see count_errors()'s docstring for why that conversion needs real target-city data and
        # isn't itself a zero-shot number; the share-based MAE/RMSE/sMAPE/JSD/spearman above it
        # are the actual zero-shot result.
        type2_results[key] = {
            "uniform_allocation": score(np.broadcast_to(uniform_share, share_tgt.shape), share_tgt, "uniform allocation", totals_tgt),
            "climatology_transfer": score(clim[hours_tgt], share_tgt, f"copy {src}'s pattern onto {tgt}", totals_tgt),
            "xgboost_alone_no_mae": score(pred_xgb_raw, share_tgt, "XGBoost alone (no MAE)", totals_tgt),
            "mae_plus_xgboost": score(pred_xgb_mae, share_tgt, "Raw clues + MAE embedding (proposed)", totals_tgt),
        }
        # Kept for the spatial-map / error-map / feature-importance cells below.
        type2_raw[key] = {
            "src": src, "tgt": tgt,
            "share_tgt": share_tgt, "pred_mae": pred_xgb_mae, "pred_raw": pred_xgb_raw,
            "hours_tgt": hours_tgt, "days_tgt": days_tgt, "totals_tgt": totals_tgt,
            "xgb_mae": xgb_mae, "xgb_raw": xgb_raw,
            "src_static": src_static, "src_embed": src_embed, "src_mae_features": src_mae_features,
            "embed_dim": src_embed.shape[-1],
        }
    except Exception as e:
        type2_results[f"{src}_to_{tgt}"] = {"error": f"{type(e).__name__}: {e}"}
        print(f"LOSO {src}->{tgt} FAILED -- {type(e).__name__}: {e}")

print("\nHow to read this: compare 'MAE + XGBoost' against 'XGBoost alone' (does the MAE step")
print("actually help?) and against 'climatology transfer' (does learning anything beat just")
print("copying the other city's pattern?). Do this for both directions above.")

### B.8b â€” San Francisco: MAE-only diversity check (no demand data, never used by XGBoost)

San Francisco has no taxi/rideshare demand dataset in this pipeline -- it exists **only** to widen
what the MAE step sees during its masked-reconstruction pretraining (the same `ClueMAE` /
`train_clue_mae()` used for every LOSO city above). SF never enters `CITY_DEMAND`, is never a
`src`/`tgt` in the B.8 loop, and XGBoost never trains or evaluates on it.

SF's public land-use parcel dataset (`data.sf.gov`, an archived 2023 snapshot) is used for exactly
one thing: sampling parcel centroids to derive a bounding box for the city. Once that box exists,
the actual static clues come from the same generic `build_static_features()` / OpenStreetMap path
every other city already uses -- SF's land-use categories themselves are not fed into the model,
only its geographic extent.

In [ ]:
# --- B.8b setup: San Francisco bounding box from public land-use parcels (geometry only) ---
SF_LANDUSE_URL = "https://data.sf.gov/resource/fdfd-xptc.json"


def first_point_from_geom(geom):
    """One representative [lon, lat] point per parcel polygon -- cheap extraction, no full
    shapely/geopandas geometry processing needed just to estimate a bounding box."""
    try:
        if geom["type"] == "MultiPolygon":
            return tuple(geom["coordinates"][0][0][0])
        elif geom["type"] == "Polygon":
            return tuple(geom["coordinates"][0][0])
        return None
    except Exception:
        return None


STATIC_SF_PATH = "static_sf.npz"
DOWNTOWN_QUERY["sf"] = "Downtown San Francisco, CA, USA"

# Both the SF parcel sample (20k Socrata records) and the OSM POI queries build_static_features()
# makes are worth skipping on repeat runs -- SF has no demand data, so its static clues depend on
# nothing but its fixed bounding box, exactly like the other cities' cache above.
_static_sf_cache = find_precomputed(STATIC_SF_PATH)
if _static_sf_cache:
    _npz_sf = np.load(_static_sf_cache)
    STATIC_SF = _npz_sf["STATIC_SF"]
    sf_n_parcels_sampled = int(_npz_sf["n_parcels_sampled"]) if "n_parcels_sampled" in _npz_sf else None
    print(f"San Francisco static clues: reusing cached {_static_sf_cache} -- skipping parcel sample + OSM queries.")
else:
    print("San Francisco: sampling land-use parcels to establish bounding box...")
    sf_records = []
    offset = 0
    while len(sf_records) < 20_000:
        resp = requests.get(SF_LANDUSE_URL, params={"$select": "the_geom", "$limit": 5000, "$offset": offset}, timeout=60)
        resp.raise_for_status()
        batch = resp.json()
        if not batch:
            break
        sf_records.extend(batch)
        offset += 5000
    print(f"  fetched {len(sf_records)} parcel records")

    sf_points = [first_point_from_geom(r["the_geom"]) for r in sf_records if r.get("the_geom")]
    sf_points = np.array([p for p in sf_points if p is not None], dtype=np.float64)

    sf_lon_lo, sf_lon_hi = np.percentile(sf_points[:, 0], [0.5, 99.5])
    sf_lat_lo, sf_lat_hi = np.percentile(sf_points[:, 1], [0.5, 99.5])
    sf_lat_edges, sf_lon_edges = fine_grid_edges(sf_lat_lo, sf_lat_hi, sf_lon_lo, sf_lon_hi)
    print(f"SF bbox from {len(sf_points)} sampled parcels: lat[{sf_lat_lo:.4f},{sf_lat_hi:.4f}] lon[{sf_lon_lo:.4f},{sf_lon_hi:.4f}]")

    sf_n_parcels_sampled = len(sf_points)

    print("Building San Francisco static clues (map-only, normalized units)...")
    STATIC_SF = build_static_features("sf", sf_lat_edges, sf_lon_edges)
    np.savez(STATIC_SF_PATH, STATIC_SF=STATIC_SF, n_parcels_sampled=sf_n_parcels_sampled)

assert N_STATIC == STATIC_SF.shape[-1]

In [ ]:
# --- B.8b: MAE reconstruction on SF, reusing the exact same MAE class/training function as
# every LOSO city above -- no XGBoost, no demand, purely a diversity/robustness check on the
# masked-reconstruction step itself. ---
print("Training MAE on San Francisco's static clues (MAE-only diversity check -- no demand, no XGBoost)...")
sf_mae_model, sf_embed = train_clue_mae(STATIC_SF, epochs=300)

X_sf = torch.tensor(STATIC_SF.reshape(-1, STATIC_SF.shape[-1]), device=device)
sf_mae_model.eval()
with torch.no_grad():
    recon_sf, _ = sf_mae_model(X_sf)
    sf_recon_mse = float(((recon_sf - X_sf) ** 2).mean())
print(f"San Francisco MAE-only reconstruction MSE (full clues, no masking): {sf_recon_mse:.5f}")

mae_only_results = {
    "san_francisco": {
        "reconstruction_mse": sf_recon_mse,
        "embed_dim": int(sf_embed.shape[-1]),
        "n_parcels_sampled": sf_n_parcels_sampled,
    },
}

### B.8c — Where does the error come from, and does the model pick the right cells?

Two questions the headline MAE cannot answer, both computed on the Type II predictions above
(evaluation only -- nothing new is trained here):

1. **Busy vs quiet cells.** The top 5% of cells by real demand carry most of the trips, so they
   dominate MAE_trips. Reported separately, next to a uniform-allocation reference so the size of
   the model's advantage is visible.
2. **Ranking.** Does the model's list of busiest cells match the real one? `top-k overlap` counts
   how many of the model's k busiest cells are truly among the k busiest, and
   `rank_of_true_busiest` is where the model places the single busiest real cell (1 = perfect).
   This is a separate question from getting the magnitude right, and it is the one that matters:
   sharpening the predictions made busy-cell error *worse*, which is only possible when the cells
   being emphasised are the wrong ones.


In [ ]:
# --- B.8c: busy-vs-quiet error and ranking quality, on the Type II predictions ---
type2_diagnostics = {}

for key, raw in type2_raw.items():
    share_tgt, totals_tgt = raw["share_tgt"], raw["totals_tgt"]
    uniform = np.full_like(share_tgt, 1.0 / N_CELLS)

    type2_diagnostics[key] = {}
    for name, pred in [("uniform_allocation", uniform),
                       ("xgboost_alone_no_mae", raw["pred_raw"]),
                       ("mae_plus_xgboost", raw["pred_mae"])]:
        entry = {"busy_quiet": stratified_trip_errors(pred, share_tgt, totals_tgt)}
        if name != "uniform_allocation":          # ranking is meaningless for a constant prediction
            entry["ranking"] = ranking_diagnostics(pred, share_tgt, totals_tgt)
        type2_diagnostics[key][name] = entry

    u = type2_diagnostics[key]["uniform_allocation"]["busy_quiet"]
    m = type2_diagnostics[key]["mae_plus_xgboost"]
    rk = m["ranking"]
    print(f"{key:18s} busy MAE_trips: uniform={u['MAE_trips_busy']:8.3f} model={m['busy_quiet']['MAE_trips_busy']:8.3f} | "
          f"quiet: uniform={u['MAE_trips_quiet']:6.3f} model={m['busy_quiet']['MAE_trips_quiet']:6.3f} | "
          f"top10={rk['top10_overlap']}/10 top50={rk['top50_overlap']}/50 true-busiest ranked #{rk['rank_of_true_busiest']}")


### B.8d — Native-resolution evaluation (NYC and Chicago)

Chicago and NYC identify each trip only by its census tract / taxi zone. The 2,000-cell "truth" for
those cities is therefore partly our own assumption: a zone's demand is distributed across its cells
by area, so *which cell inside a zone* is right is not something the data can verify. Porto carries
real per-trip GPS, so for Porto the grid cell already is the native resolution.

Here the cells are merged back into the zones/tracts they came from (a border cell is split by how much
of it each zone covers) -- identically for the real and the predicted map -- and scored again. Each
result sits next to a uniform-by-area guess, and `model_vs_uniform_MAE_ratio` compares how much better
than that guess the model is at the zone level vs the cell level. If the model looks much better
against uniform at zone level, part of its cell-level error is within-zone detail nobody can verify.

Aggregating always shrinks errors, so zone-level MAE must never be compared numerically with
cell-level MAE, or across cities: use the ratios and the normalised errors (`nMAE`, `nRMSE` = error
divided by the average real demand per unit).


In [ ]:
# --- B.8d: native-resolution evaluation for the two cities identified by zone, not by coordinate ---
import zipfile, glob, traceback

native_eval = {}     # per directed pair; written to results_summary.json
native_series = {}   # a few short arrays kept only for the charts below

NATIVE_UNITS = {"nyc": "taxi zones", "chicago": "census tracts"}
CHI_TRACTS_EVAL_URL = "https://data.cityofchicago.org/resource/74p9-q2aq.geojson?$limit=5000"
NATIVE_WINDOW = 336   # hours plotted in the charts


def load_native_units(city):
    """Zone polygons (WGS84) and readable names for a city whose feed identifies trips by zone."""
    import geopandas as gpd
    if city == "nyc":
        r = requests.get(NYC_ZONES_URL, timeout=180)
        r.raise_for_status()
        with open("taxi_zones_eval.zip", "wb") as f:
            f.write(r.content)
        with zipfile.ZipFile("taxi_zones_eval.zip") as zf:
            zf.extractall("taxi_zones_eval")
        g = gpd.read_file(glob.glob("taxi_zones_eval/**/*.shp", recursive=True)[0]).to_crs(epsg=4326)
        return g, [str(z) for z in g["zone"]]
    r = requests.get(CHI_TRACTS_EVAL_URL, timeout=300)
    r.raise_for_status()
    with open("chicago_tracts_eval.geojson", "wb") as f:
        f.write(r.content)
    g = gpd.read_file("chicago_tracts_eval.geojson").to_crs(epsg=4326)
    return g, [f"tract {n} (community area {c})" for n, c in zip(g["name10"], g["commarea_n"])]


def units_to_cells_weights(gdf, lat_e, lon_e):
    """B[z, c] = share of cell c's zone-covered area that belongs to zone z (covered columns sum to 1).
    A zone's value is then B @ (cell values), for the real and the predicted map alike."""
    W = zone_area_matrix(list(gdf.geometry.values), lat_e, lon_e)             # zone -> cells, rows sum to 1
    area = gdf.to_crs(gdf.estimate_utm_crs()).geometry.area.values             # m^2, so relative areas are right
    A = W * area[:, None]                                                      # zone area falling in each cell
    col = A.sum(axis=0)
    return np.divide(A, col[None, :], out=np.zeros_like(A), where=col[None, :] > 0).astype(np.float32)


def level_metrics(pred, true, totals):
    """MAE/RMSE in trips per unit-hour, and the same divided by the average real demand per unit."""
    err = (pred - true) * totals[:, None]
    mean_true = float((true.T @ totals).sum() / (len(totals) * true.shape[1]))
    mae = float(np.abs(err).mean(dtype=np.float64))
    rmse = float(np.sqrt((err.astype(np.float64) ** 2).mean()))
    return {"MAE_trips": mae, "RMSE_trips": rmse, "nMAE": mae / mean_true, "nRMSE": rmse / mean_true}


def native_eval_pair(raw, B, names):
    share, tot = raw["share_tgt"], raw["totals_tgt"]
    n_cells = share.shape[1]
    true_u = share @ B.T                                                       # (hours, units)
    preds = {
        "uniform_by_area": np.broadcast_to((B.sum(axis=1) / n_cells)[None, :], true_u.shape),
        "xgboost_alone_no_mae": raw["pred_raw"] @ B.T,
        "mae_plus_xgboost": raw["pred_mae"] @ B.T,
    }
    out = {"n_units": int(B.shape[0])}
    for name, p in preds.items():
        out[name] = level_metrics(p, true_u, tot)

    uni_cell = np.full(share.shape, 1.0 / n_cells, dtype=np.float32)
    cell = {"uniform": level_metrics(uni_cell, share, tot),
            "mae_plus_xgboost": level_metrics(raw["pred_mae"], share, tot)}
    out["cell_level_reference"] = cell
    out["model_vs_uniform_MAE_ratio"] = {
        "zone": out["mae_plus_xgboost"]["MAE_trips"] / out["uniform_by_area"]["MAE_trips"],
        "cell": cell["mae_plus_xgboost"]["MAE_trips"] / cell["uniform"]["MAE_trips"],
    }

    pm = preds["mae_plus_xgboost"]
    bq = stratified_trip_errors(pm, true_u, tot)             # generic in the number of columns
    out["busy_quiet"] = {k.replace("cells", "units"): v for k, v in bq.items()}
    out["ranking"] = ranking_diagnostics(pm, true_u, tot)

    mean_true_u = (true_u.T @ tot) / len(tot)
    mean_pred_u = (pm.T @ tot) / len(tot)
    act = mean_true_u > 0
    out["spearman_mean_demand"] = float(spearmanr(mean_true_u[act], mean_pred_u[act]).correlation)

    order = np.argsort(-mean_true_u)
    pred_rank = np.empty(len(order), dtype=int)
    pred_rank[np.argsort(-mean_pred_u)] = np.arange(1, len(order) + 1)
    out["top_real_units"] = [
        {"unit": names[z], "real_mean_trips_per_hour": float(mean_true_u[z]),
         "predicted_mean_trips_per_hour": float(mean_pred_u[z]), "predicted_rank": int(pred_rank[z])}
        for z in order[:10]]

    z0 = order[0]
    series = {"name": names[z0],
              "real": (true_u[:NATIVE_WINDOW, z0] * tot[:NATIVE_WINDOW]).astype(np.float64),
              "pred": (pm[:NATIVE_WINDOW, z0] * tot[:NATIVE_WINDOW]).astype(np.float64)}
    return out, series


_native_cache = {}
for key, raw in type2_raw.items():
    tgt = raw["tgt"]
    if tgt not in NATIVE_UNITS:
        native_eval[key] = {"unit": "grid cells",
                            "note": "Porto's feed carries real per-trip GPS, so its native resolution is the grid "
                                    "itself -- see type2_zero_shot and type2_diagnostics."}
        continue
    try:
        if tgt not in _native_cache:
            gdf, names = load_native_units(tgt)
            le, lo = (nyc_lat_edges, nyc_lon_edges) if tgt == "nyc" else (lat_edges, lon_edges)
            B = units_to_cells_weights(gdf, le, lo)
            _native_cache[tgt] = (B, names)
            print(f"[{tgt}] {B.shape[0]} {NATIVE_UNITS[tgt]}; {(B.sum(axis=0) > 0).sum()} of {B.shape[1]} cells are covered")
        B, names = _native_cache[tgt]
        out, series = native_eval_pair(raw, B, names)
        out["unit"] = NATIVE_UNITS[tgt]
        native_eval[key] = out
        native_series[key] = series
        rk = out["ranking"]
        print(f"{key:18s} [{NATIVE_UNITS[tgt]}] MAE_trips: uniform={out['uniform_by_area']['MAE_trips']:.3f} "
              f"model={out['mae_plus_xgboost']['MAE_trips']:.3f} | model/uniform: zone={out['model_vs_uniform_MAE_ratio']['zone']:.2f} "
              f"cell={out['model_vs_uniform_MAE_ratio']['cell']:.2f} | top10 overlap={rk['top10_overlap']}/10 "
              f"busiest ranked #{rk['rank_of_true_busiest']} | spearman={out['spearman_mean_demand']:.2f}")
    except Exception as e:
        traceback.print_exc()
        native_eval[key] = {"error": f"{type(e).__name__}: {e}"}


### B.8e — Does the model reproduce how concentrated demand is?

Lower MAE can come from genuinely better predictions or from simply flattening them, so this
section measures concentration directly, per pair:

* **Concentration of the real map, the predicted map, and the source city's map** (share of all trips
  held by the busiest cell / top 5 / top 20, plus Gini and normalised entropy).
* **Mass ratio on the real top-k cells** -- of the trips that really happened in the k busiest cells,
  what fraction did the model put there? 1.0 is perfect; well below 1 means the peaks are being
  under-filled.
* **Peak-share ratio** -- the model's busiest-cell share in an hour divided by the real one, averaged
  over hours, regardless of *which* cell it is. This isolates peak height from cell choice.


In [ ]:
# --- B.8e: concentration diagnostics ---
concentration_diagnostics = {}


def concentration_stats(whole_period_share):
    """Concentration of a map given as each cell's share of all trips over the whole period."""
    p = np.asarray(whole_period_share, dtype=np.float64)
    p = p / p.sum()
    s = np.sort(p)
    n = len(p)
    gini = float((2.0 * (np.arange(1, n + 1) * s).sum()) / (n * s.sum()) - (n + 1) / n)
    nz = p[p > 0]
    entropy = float(-(nz * np.log(nz)).sum() / np.log(n))          # 1 = perfectly flat, 0 = one cell
    top = s[::-1]
    return {"top1_share": float(top[0]), "top5_share": float(top[:5].sum()),
            "top20_share": float(top[:20].sum()), "gini": gini, "entropy_normalised": entropy}


def mass_ratio_on_true_topk(pred, true, totals, ks=(1, 5, 10, 20, 50)):
    """Trips the model placed in the real top-k cells / trips that really happened there."""
    true_mass, pred_mass = true.T @ totals, pred.T @ totals
    order = np.argsort(-true_mass)
    return {f"top{k}": float(pred_mass[order[:k]].sum() / true_mass[order[:k]].sum()) for k in ks}


for key, raw in type2_raw.items():
    try:
        share, tot = raw["share_tgt"], raw["totals_tgt"]
        src_demand = CITY_DEMAND[raw["src"]]
        src_total = src_demand.reshape(src_demand.shape[0] * src_demand.shape[1], -1).sum(axis=0)
        entry = {"source_city": raw["src"], "source": concentration_stats(src_total),
                 "truth": concentration_stats(share.T @ tot)}
        for name, pred in [("xgboost_alone_no_mae", raw["pred_raw"]), ("mae_plus_xgboost", raw["pred_mae"])]:
            entry[name] = {
                "prediction": concentration_stats(pred.T @ tot),
                "mass_ratio_on_true_topk": mass_ratio_on_true_topk(pred, share, tot),
                "peak_share_ratio": float(pred.max(axis=1).mean() / share.max(axis=1).mean()),
            }
        concentration_diagnostics[key] = entry
        m = entry["mae_plus_xgboost"]
        print(f"{key:18s} top-20 share: source={entry['source']['top20_share']:.2f} real={entry['truth']['top20_share']:.2f} "
              f"predicted={m['prediction']['top20_share']:.2f} | mass on real top-20={m['mass_ratio_on_true_topk']['top20']:.2f} "
              f"| peak-share ratio={m['peak_share_ratio']:.2f}")
    except Exception as e:
        traceback.print_exc()
        concentration_diagnostics[key] = {"error": f"{type(e).__name__}: {e}"}


### B.8f — Real vs predicted, beyond the single busiest cell

The earlier charts showed only the busiest cell, the hardest possible view. For every directed pair:

* `cells/` -- 14 days of real vs predicted at cells ranked 1, 10, 50, 200 and 600 **among the active
  cells** (those averaging at least 1% of the citywide per-cell mean demand, so slivers of a large
  zone's demand don't count; NYC and Chicago have only about 1,050 cells with any demand at all).
* `topk_scatter/` -- the summed demand of the real top-1 / top-10 / top-50 cells over 14 days
  (separates *how much* from *which cells*), a log-log scatter of every active cell's average real vs
  predicted demand (on the diagonal is perfect; a flat cloud means the predictions are compressed),
  and the busiest native zone/tract for NYC and Chicago targets.


In [ ]:
# --- B.8f: fuller real-vs-predicted charts ---
import matplotlib.pyplot as plt

CHART_ROOT = "urbanmind_results/diagnostic_charts"
os.makedirs(f"{CHART_ROOT}/cells", exist_ok=True)
os.makedirs(f"{CHART_ROOT}/topk_scatter", exist_ok=True)
CHART_RANKS = (1, 10, 50, 200, 600)
chart_status = {}

for key, raw in type2_raw.items():
    try:
        share, pred, tot = raw["share_tgt"], raw["pred_mae"], raw["totals_tgt"]
        w = min(NATIVE_WINDOW, len(tot))
        x = np.arange(w)
        mean_true = (share.T @ tot) / len(tot)                       # average real trips/hour per cell
        mean_pred = (pred.T @ tot) / len(tot)
        # "active" = averaging at least 1% of the citywide per-cell mean, so cells that only receive a
        # sliver of some large zone's demand (1e-5 trips/hour) don't stretch the axes or fill the ranks
        active = np.where(mean_true >= 0.01 * mean_true.mean())[0]
        order = active[np.argsort(-mean_true[active])]               # active cells, busiest first
        ranks = [r for r in CHART_RANKS if r <= len(order)]
        tgt = raw["tgt"]

        # ---- figure 1: cells at several ranks ----
        fig, axes = plt.subplots(len(ranks), 1, figsize=(12, 2.4 * len(ranks)), sharex=True)
        for ax, r in zip(np.atleast_1d(axes), ranks):
            c = int(order[r - 1])
            ax.plot(x, share[:w, c] * tot[:w], color="steelblue", lw=1.1, label="real")
            ax.plot(x, pred[:w, c] * tot[:w], color="darkorange", lw=1.1, label="predicted")
            ax.set_title(f"{tgt}: rank {r} of {len(order)} active cells (row {c // GRID_COLS}, col {c % GRID_COLS}) -- "
                         f"average real demand {mean_true[c]:.1f} trips/h", fontsize=10)
            ax.set_ylabel("trips/hour")
            ax.legend(loc="upper right", fontsize=8)
        np.atleast_1d(axes)[-1].set_xlabel("hour index (first 14 days of the evaluation period)")
        fig.suptitle(f"Real vs predicted at several cell ranks -- trained on {raw['src']}, zero-shot on {tgt} ({key})")
        fig.tight_layout()
        fig.savefig(f"{CHART_ROOT}/cells/{key}.png", dpi=120, bbox_inches="tight")
        plt.close(fig)

        # ---- figure 2: top-k aggregates + scatter + busiest native unit ----
        fig = plt.figure(figsize=(15, 9))
        gs = fig.add_gridspec(3, 2, width_ratios=[1.6, 1])
        for i, k in enumerate((1, 10, 50)):
            ax = fig.add_subplot(gs[i, 0])
            cells_k = order[:k]
            ax.plot(x, share[:w][:, cells_k].sum(axis=1) * tot[:w], color="steelblue", lw=1.1, label="real")
            ax.plot(x, pred[:w][:, cells_k].sum(axis=1) * tot[:w], color="darkorange", lw=1.1, label="predicted")
            ax.set_title(f"total demand of the real top-{k} cell{'s' if k > 1 else ''}", fontsize=10)
            ax.set_ylabel("trips/hour")
            ax.legend(loc="upper right", fontsize=8)
        ax.set_xlabel("hour index (first 14 days)")

        ax = fig.add_subplot(gs[0:2, 1])
        ax.scatter(mean_true[active], mean_pred[active], s=7, alpha=0.45, color="steelblue")
        b = int(order[0])
        ax.scatter([mean_true[b]], [mean_pred[b]], s=45, color="crimson", label="busiest cell")
        lo = min(mean_true[active].min(), mean_pred[active].min())
        hi = max(mean_true[active].max(), mean_pred[active].max())
        ax.plot([lo, hi], [lo, hi], color="black", lw=1, ls="--", label="perfect")
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlabel("average real trips/hour"); ax.set_ylabel("average predicted trips/hour")
        ax.set_title(f"every active cell ({len(active)})", fontsize=10)
        ax.legend(fontsize=8)

        ax = fig.add_subplot(gs[2, 1])
        s = native_series.get(key)
        if s is not None:
            ax.plot(x, s["real"][:w], color="steelblue", lw=1.1, label="real")
            ax.plot(x, s["pred"][:w], color="darkorange", lw=1.1, label="predicted")
            ax.set_title(f"busiest {NATIVE_UNITS[tgt][:-1]}: {s['name'][:40]}", fontsize=10)
            ax.set_xlabel("hour index"); ax.set_ylabel("trips/hour")
            ax.legend(loc="upper right", fontsize=8)
        else:
            ax.axis("off")
            ax.text(0.5, 0.5, "native resolution = the grid cell\n(real per-trip GPS)", ha="center", va="center")
        fig.suptitle(f"Top-k aggregates, scatter and native unit -- trained on {raw['src']}, zero-shot on {tgt} ({key})")
        fig.tight_layout()
        fig.savefig(f"{CHART_ROOT}/topk_scatter/{key}.png", dpi=120, bbox_inches="tight")
        plt.close(fig)
        chart_status[key] = "ok"
        print(f"saved charts for {key}")
    except Exception as e:
        traceback.print_exc()
        chart_status[key] = f"{type(e).__name__}: {e}"


### B.9 â€” Diagnostics: real-vs-predicted over time, error maps, feature importance

Three ways to look *inside* the "MAE + XGBoost" zero-shot result from B.8, instead of just the
single aggregate sMAPE/MAE number:

1. **Real vs. predicted over time** â€” for the target city's busiest cell (by mean real demand),
   predicted vs. real trips/hour plotted against hour index: the full evaluation period, plus a
   zoomed first-2-week window. A city-wide aggregate would be a trivial match instead of a real
   test â€” `predict_all()`'s per-hour shares sum to 1 by construction, so summing predicted counts
   over every cell always reproduces the real citywide total exactly. The busiest single cell is
   where the model's spatial allocation is actually being tested over time.
2. **Feature importance** â€” which static-clue / embedding / calendar columns the frozen XGBoost
   models actually lean on, via SHAP (falls back to XGBoost's built-in gain importance if the `shap`
   package isn't available).
3. **Error maps** â€” per-cell mean absolute error and per-cell signed bias (over- vs under-prediction),
   so systematically mis-served zones are visible instead of averaged away.

All three reuse `type2_raw` populated at the end of the B.8 loop above -- rerun B.8 first if you've
restarted the kernel.

In [ ]:
import matplotlib.pyplot as plt

TEMPORAL_DIR = "urbanmind_results/temporal"
os.makedirs(TEMPORAL_DIR, exist_ok=True)

ZOOM_HOURS = 336  # first 2 weeks of the evaluation series

for key, raw in type2_raw.items():
    share_tgt, pred_mae, totals_tgt = raw["share_tgt"], raw["pred_mae"], raw["totals_tgt"]

    pred_counts = pred_mae * totals_tgt[:, None]     # trips/hour per cell, predicted
    true_counts = share_tgt * totals_tgt[:, None]    # trips/hour per cell, real

    # Busiest cell in the target city (by mean real demand) -- a single fixed cell is the
    # meaningful comparison here: summed over ALL cells, predicted always equals real exactly
    # (predict_all()'s shares sum to 1 by construction), so a citywide-total plot would be
    # trivial rather than a real test of the model.
    busiest_cell = int(np.argmax(true_counts.mean(axis=0)))
    row, col = busiest_cell // GRID_COLS, busiest_cell % GRID_COLS
    real_series = true_counts[:, busiest_cell]
    pred_series = pred_counts[:, busiest_cell]
    hour_index = np.arange(len(real_series))

    fig, axes = plt.subplots(2, 1, figsize=(12, 6))
    axes[0].plot(hour_index, real_series, label="real", color="steelblue", linewidth=0.7)
    axes[0].plot(hour_index, pred_series, label="predicted", color="darkorange", linewidth=0.7, alpha=0.85)
    axes[0].set_title(f"{raw['tgt']} busiest cell (row={row}, col={col}): full evaluation period")
    axes[0].set_xlabel("hour index (chronological)")
    axes[0].set_ylabel("trips/hour")
    axes[0].legend()

    zoom = min(ZOOM_HOURS, len(real_series))
    axes[1].plot(hour_index[:zoom], real_series[:zoom], label="real", color="steelblue", linewidth=1.1)
    axes[1].plot(hour_index[:zoom], pred_series[:zoom], label="predicted", color="darkorange", linewidth=1.1, alpha=0.85)
    axes[1].set_title(f"{raw['tgt']} busiest cell: first {zoom} hours (zoomed)")
    axes[1].set_xlabel("hour index (chronological)")
    axes[1].set_ylabel("trips/hour")
    axes[1].legend()

    fig.suptitle(f"Real vs predicted, busiest {raw['tgt']} cell -- trained on {raw['src']}, zero-shot ({key})")
    fig.tight_layout()
    out_path = f"{TEMPORAL_DIR}/{key}.png"
    fig.savefig(out_path, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"Saved {out_path}")


In [ ]:
STATIC_NAMES = list(POI_GROUPS.keys()) + [
    "dist_downtown_rel", "dist_transit_rel", "dist_water_rel", "busyness_density", "mix_score",
    "downtown_missing", "transit_missing", "water_missing",
]  # 16 columns, matches V23 build_static_features()
CAL_NAMES = [
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "weekend", "holiday",
    "month_sin", "month_cos", "rush_hour",
]  # 9 columns, matches calendar_row()'s order

try:
    import shap
    HAVE_SHAP = True
except ImportError:
    HAVE_SHAP = False
    print("`shap` not installed -- falling back to XGBoost's built-in gain importance.")

IMPORTANCE_DIR = "urbanmind_results/feature_importance"
os.makedirs(IMPORTANCE_DIR, exist_ok=True)
SAMPLE_ROWS = 20_000  # subsample for SHAP -- it's O(rows) but no need for the full multi-million-row set

def importance_for(model, feature_grid, feature_names, src_city, day_offset, tag, key):
    """feature_grid: (GRID_ROWS, GRID_COLS, F) static/embedding clues for the SOURCE city
    (the city the model was actually trained on -- importance is a property of the trained model,
    so it must be measured on training-distribution inputs, not the target city's)."""
    X, _y = build_training_rows(CITY_DEMAND[src_city], feature_grid,
                                 day_offset_start=day_offset, max_rows=SAMPLE_ROWS, city=src_city)
    if len(X) > SAMPLE_ROWS:
        rng = np.random.default_rng(0)
        X = X[rng.choice(len(X), size=SAMPLE_ROWS, replace=False)]

    if HAVE_SHAP:
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X)
        importance = np.abs(sv).mean(axis=0)
        ylabel = "mean |SHAP value|"
    else:
        importance = model.feature_importances_  # gain-based for the sklearn XGBRegressor API
        ylabel = "gain importance"

    order = np.argsort(importance)[::-1]
    fig, ax = plt.subplots(figsize=(7, max(3, 0.32 * len(feature_names))))
    ax.barh([feature_names[i] for i in order][::-1], importance[order][::-1], color="steelblue")
    ax.set_xlabel(ylabel)
    ax.set_title(f"{tag} -- feature importance ({key}, trained on {src_city})")
    fig.tight_layout()
    out_path = f"{IMPORTANCE_DIR}/{key}_{tag}.png"
    fig.savefig(out_path, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"Saved {out_path}")
    return dict(zip(feature_names, importance.tolist()))

feature_importance_results = {}
for key, raw in type2_raw.items():
    src = raw["src"]
    embed_names = [f"embed_{i}" for i in range(raw["embed_dim"])]

    feature_importance_results[f"{key}_xgboost_alone"] = importance_for(
        raw["xgb_raw"], raw["src_static"], STATIC_NAMES + CAL_NAMES,
        src, CITY_D0[src], "xgboost_alone_no_mae", key)

    feature_importance_results[f"{key}_mae_plus_xgboost"] = importance_for(
        raw["xgb_mae"], raw["src_mae_features"], STATIC_NAMES + embed_names + CAL_NAMES,
        src, CITY_D0[src], "mae_plus_xgboost", key)

print("\nHow to read this: if calendar columns (hour_sin/cos, weekend, rush_hour, ...) dominate over")
print("the static/embedding columns, the model is mostly learning generic time-of-day patterns rather")
print("than transferring anything about *where* demand concentrates -- worth knowing before claiming")
print("the static clues / MAE embeddings are what makes cross-city transfer work.")


In [ ]:
ERROR_MAP_DIR = "urbanmind_results/error_maps"
os.makedirs(ERROR_MAP_DIR, exist_ok=True)

error_map_stats = {}
for key, raw in type2_raw.items():
    share_tgt, pred_mae, totals_tgt = raw["share_tgt"], raw["pred_mae"], raw["totals_tgt"]

    pred_counts = pred_mae * totals_tgt[:, None]
    true_counts = share_tgt * totals_tgt[:, None]
    err = pred_counts - true_counts                      # signed, trips/cell/hour
    abs_err_per_cell = np.abs(err).mean(axis=0)           # (N_CELLS,) mean absolute error per cell
    bias_per_cell = err.mean(axis=0)                      # (N_CELLS,) signed: + = over-predicted, - = under

    mae_grid = abs_err_per_cell.reshape(GRID_ROWS, GRID_COLS)
    bias_grid = bias_per_cell.reshape(GRID_ROWS, GRID_COLS)
    bias_clip = np.abs(bias_grid).max()

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    im0 = axes[0].imshow(mae_grid, cmap="inferno")
    axes[0].set_title(f"{raw['tgt']}: per-cell MAE (trips/hour)\n{key}")
    axes[0].set_xticks([]); axes[0].set_yticks([])
    fig.colorbar(im0, ax=axes[0], shrink=0.85)

    im1 = axes[1].imshow(bias_grid, cmap="RdBu_r", vmin=-bias_clip, vmax=bias_clip)
    axes[1].set_title(f"{raw['tgt']}: signed bias (+over / -under)\n{key}")
    axes[1].set_xticks([]); axes[1].set_yticks([])
    fig.colorbar(im1, ax=axes[1], shrink=0.85)

    fig.tight_layout()
    out_path = f"{ERROR_MAP_DIR}/{key}.png"
    fig.savefig(out_path, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"Saved {out_path}")

    top_over = np.argsort(bias_per_cell)[::-1][:5]
    top_under = np.argsort(bias_per_cell)[:5]
    error_map_stats[key] = {
        "mean_abs_error_per_cell": float(abs_err_per_cell.mean()),
        "top5_overpredicted_cells_rc": [(int(c // GRID_COLS), int(c % GRID_COLS)) for c in top_over],
        "top5_underpredicted_cells_rc": [(int(c // GRID_COLS), int(c % GRID_COLS)) for c in top_under],
    }
    mae_pc = error_map_stats[key]["mean_abs_error_per_cell"]
    over_pc = error_map_stats[key]["top5_overpredicted_cells_rc"]
    under_pc = error_map_stats[key]["top5_underpredicted_cells_rc"]
    print(f"{key}: mean per-cell MAE = {mae_pc:.4f} trips/hour")
    print(f"  most over-predicted cells (row, col): {over_pc}")
    print(f"  most under-predicted cells (row, col): {under_pc}")


### B.5 â€” Package everything into a downloadable zip

Run this notebook with **Save & Run All (Commit)** (not just the interactive "Run All") so it keeps running in the background even if you close the browser tab â€” see the step-by-step below. When it finishes, everything this cell writes shows up under the notebook's **Output** tab as a downloadable zip.

This does **not** try to zip the raw multi-year trip data (that would be huge and Kaggle output has size limits) â€” it saves the things needed to interpret results afterward: all the numbers (losses, region split, config), plus small sample chunks of the built city grids, plus the trained Part-B model weights.

In [ ]:
# --- V23 ablation chart: strict zero-shot spatial-share results ---
ABLATION_DIR = "urbanmind_results/ablation"
os.makedirs(ABLATION_DIR, exist_ok=True)

rows = []
for direction, methods in type2_results.items():
    for method, metrics in methods.items():
        if "error" not in metrics:
            rows.append({"direction": direction, "method": method, **metrics})
ablation_table = pd.DataFrame(rows)
ablation_table.to_csv(f"{ABLATION_DIR}/zero_shot_ablation.csv", index=False)

plot_methods = ["xgboost_alone_no_mae", "mae_plus_xgboost"]
plot = ablation_table[ablation_table["method"].isin(plot_methods)].copy()
labels = {"xgboost_alone_no_mae": "XGBoost only", "mae_plus_xgboost": "Raw + MAE embedding"}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, metric, title in zip(axes, ["MAE", "JSD"], ["Spatial-share MAE (lower is better)", "Jensen-Shannon divergence (lower is better)"]):
    pivot = plot.pivot(index="direction", columns="method", values=metric).reindex(columns=plot_methods)
    pivot.rename(columns=labels).plot.bar(ax=ax, color=["#4C78A8", "#F58518"], rot=0)
    ax.set_title(title)
    ax.set_xlabel("transfer direction")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(title="model")
fig.suptitle("V23 strict zero-shot ablation: MAE must beat the raw-clue baseline")
fig.tight_layout()
chart_path = f"{ABLATION_DIR}/zero_shot_ablation.png"
fig.savefig(chart_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved {chart_path} and {ABLATION_DIR}/zero_shot_ablation.csv")


In [ ]:
import json as _json
import zipfile

RESULTS_DIR = "urbanmind_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

summary = {
    "config": {
        "GRID_ROWS": GRID_ROWS, "GRID_COLS": GRID_COLS, "TIME_SLOTS_PER_DAY": TIME_SLOTS_PER_DAY,
        "CHI_YEARS": CHI_YEARS, "NYC_YEAR_MONTHS": NYC_YEAR_MONTHS,
        "feature_contract": "V23: POI density/kmÂ²; relative distances; no airport feature; source-only robust scaling",
        "location_contract": "trip demand distributed across its zone polygon by area (Chicago=census tract, NYC=taxi zone); Porto carries real per-trip GPS",
        "PORTO_INFO": {
            "url": PORTO_ZIP_URL,
            "date_range": "2013-07-01 to 2014-06-30",
            "source": "UCI ECML/PKDD 2015 Taxi Trajectory Prediction Challenge",
        },
        "SF_INFO": {
            "url": SF_LANDUSE_URL,
            "source": "San Francisco land-use parcels (data.sf.gov, archived 2023 snapshot) -- "
                       "used ONLY to derive a bounding box for MAE-only pretraining diversity; "
                       "SF has no demand data and never appears in CITY_DEMAND or the B.8 LOSO loop.",
        },
    },
    "data_quality": {
        "eda_before_cleaning": eda_summary,  # B.2b: spikes/gaps/summary stats found in the raw data
        "cleaning_applied": {                # B.2c: what B.2c actually fixed
            "chicago": {"gap_days_filled": chi_days_filled, "spikes_capped": chi_spikes_capped},
            "nyc": {"gap_days_filled": nyc_days_filled, "spikes_capped": nyc_spikes_capped},
            "porto": {"gap_days_filled": porto_days_filled, "spikes_capped": porto_spikes_capped},
        },
    },
    "type2_zero_shot": type2_results,  # B.8 -- Type II zero-shot LOSO + its baselines, the actual result
    "mae_only_diversity": mae_only_results,  # B.8b -- MAE reconstruction quality on cities with no demand data
    "type2_diagnostics": type2_diagnostics,  # B.8c -- busy vs quiet error + ranking quality, per pair
    "native_resolution_eval": native_eval,  # B.8d -- NYC/Chicago scored at their native zones/tracts
    "concentration_diagnostics": concentration_diagnostics,  # B.8e -- does the model reproduce concentration?
}
with open(f"{RESULTS_DIR}/results_summary.json", "w") as f:
    _json.dump(summary, f, indent=2)

zip_path = "/kaggle/working/urbanmind_results.zip" if os.path.exists("/kaggle") else "urbanmind_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(RESULTS_DIR):
        for fn in files:
            fp = os.path.join(root, fn)
            zf.write(fp, arcname=os.path.relpath(fp, RESULTS_DIR))

print(f"Wrote {zip_path}")
print("After the run finishes (Save & Run All / Commit), download this zip from the notebook's")
print("Output tab -- results_summary.json inside has every number needed to interpret what")
print("happened, without needing to re-run anything.")